<!-- COMMONS LAUNCHER v4 · generated by tools/notebooks.py · do not edit by hand -->
<a href="https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials"><img src="https://raw.githubusercontent.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master/brand/synapsa-commons-badge.png" alt="Synapsa Commons" height="36"></a>

Free, hands-on AI courses that run anywhere, from the team building [Synapsa](https://synapsa.realai.eu), an AI-native
learning platform.

© 2026 RealAI · free to learn from, share and adapt, not to sell ([CC BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/)).
The notice at the end of this notebook says what you may and may not do.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/blob/master/lessons/T00-L04-living-on-free-tiers/lesson.ipynb)
[![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/blob/master/lessons/T00-L04-living-on-free-tiers/lesson.ipynb)
[![Open in Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master?labpath=lessons/T00-L04-living-on-free-tiers/lesson.ipynb)
[![Open in Codespaces](https://github.com/codespaces/badge.svg)](https://codespaces.new/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials)

This lesson needs Python 3.11 or newer. The cell below fetches the files it needs beside it,
and does nothing where they are already present. On Kaggle, switch Internet on in the
notebook's settings first; Kaggle allows that only for phone-verified accounts.

In [ ]:
# --- COMMONS LAUNCHER v4 · generated by tools/notebooks.py · do not edit by hand ---
# Makes this notebook run anywhere. Every line is a no-op when the thing is already present,
# so a local clone pays nothing and an online notebook repairs itself.
import importlib.util, os, subprocess, sys, urllib.request
from pathlib import Path

COMMONS_PIP = []            # (import name, pinned pip spec) for what this lesson imports
COMMONS_SIBLINGS = ["assets/SOURCE.md", "assets/free_tiers.json"]    # files that must sit beside the notebook
# A fork, a classroom mirror or an offline copy can serve the files from elsewhere by setting
# COMMONS_RAW_OVERRIDE before running this cell.
COMMONS_RAW = os.environ.get("COMMONS_RAW_OVERRIDE") or "https://raw.githubusercontent.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master/lessons/T00-L04-living-on-free-tiers/"

# Resolve siblings against the LESSON's own directory, not the working directory. A notebook
# has no __file__ and runs with cwd alongside itself; a grader imports this file from the repo
# root. Checking cwd blindly makes the grader think every sibling is missing and reach for the
# network -- which would put a download on a graded path.
try:
    COMMONS_DIR = Path(__file__).resolve().parent
except NameError:
    COMMONS_DIR = Path.cwd()


def commons_host() -> str:
    """Name the notebook service we are on. Used for the message, and for honest errors."""
    try:
        if importlib.util.find_spec("google.colab") is not None:
            return "Google Colab"
    except (ImportError, ValueError):
        pass
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
        return "Kaggle"
    if os.environ.get("BINDER_SERVICE_HOST"):
        return "Binder"
    if os.environ.get("CODESPACES"):
        return "GitHub Codespaces"
    return "a local Python environment"


_missing = [pip for imp, pip in COMMONS_PIP if importlib.util.find_spec(imp) is None]
if _missing:
    print("installing " + ", ".join(_missing) + " ...")
    # pip everywhere a student is likely to be; uv-managed local venvs ship without pip.
    if importlib.util.find_spec("pip") is not None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_missing], check=True)
    else:
        subprocess.run(["uv", "pip", "install", "-q", "--python", sys.executable, *_missing],
                       check=True)
    importlib.invalidate_caches()

_fetched = []
for _name in COMMONS_SIBLINGS:
    if not (COMMONS_DIR / _name).exists():
        (COMMONS_DIR / _name).parent.mkdir(parents=True, exist_ok=True)
        try:
            urllib.request.urlretrieve(COMMONS_RAW + _name, COMMONS_DIR / _name)
            _fetched.append(_name)
        except Exception as _e:  # Kaggle disables the internet by default; say so plainly
            raise RuntimeError(
                f"this lesson needs {_name} beside the notebook and could not fetch it "
                f"({_e}). On Kaggle, switch Internet on in the notebook settings panel "
                f"(Kaggle allows that only for phone-verified accounts); otherwise download it "
                f"from {COMMONS_RAW + _name} and upload it beside the notebook."
            ) from None

print("ready on " + commons_host() + ("; fetched " + ", ".join(_fetched) if _fetched else ""))
# --- END COMMONS LAUNCHER ---

# T00-L04 · Living on free tiers: a quota, rate-limit and retry planner for a class that shares one key, checked by simulation

**You will build:** the planning layer that tells a teacher whether a whole class can finish a
notebook on free API tiers. It has a token-bucket limiter, retries with jittered backoff, a
scheduler that decides whose request goes next, and a planner that turns a tier's published
limits into the number of learners who can finish. Then you will simulate a class sharing one key
and measure what the planner predicts, what retries waste, and what a quota cut does.

**Time:** ~60 minutes · **Runs on:** a laptop CPU, 8 GiB RAM, no GPU, no network, no API key
· **Prerequisites:** `T00-L01-the-8gb-track` (only for the `cpu8` tier vocabulary; nothing from
it is imported). Python classes, dictionaries and integer arithmetic.

By the end you will be able to:

1. Implement `TokenBucket`, a limiter in exact integer arithmetic that refills continuously,
   refuses a take it cannot cover in full, and reports the first millisecond a take will work.
2. Implement `backoff_ms`, capped exponential backoff with full jitter added on top of a
   server's `Retry-After`, and measure what different retry rules cost a class in wasted requests.
3. Implement `Pacer`, which holds per-minute buckets and per-day counters and returns the
   earliest time a request breaks none of them, or `None` once a daily quota is spent.
4. Implement `class_capacity`, an upper bound on how many learners can finish on a shared
   quota that names the limit that binds, and check it against a simulated class.
5. Implement `pick_next`, measure how the order a shared key serves learners in changes how
   many finish, and re-plan when one provider cuts its quota.

**What is real here and what is not.** The limits in `assets/free_tiers.json` are what each
vendor's own documentation said on the snapshot date printed below; `claims.yaml` quotes the
pages. Nothing else in this notebook is measured on a provider. It runs with no network and no
API key, so every request, every 429 and every `Retry-After` comes from a simulated provider
written for this lesson, and every count of "finishers" or "rejected requests" is a measurement
of that model, not of any real service. The model is the simplest rule that fits the published
limits: one bucket per per-minute limit (Cerebras documents buckets; the other pages do not say
how they count) and one counter per daily limit. Real services differ in ways only testing
them will show, and where the documentation is silent this lesson says so instead of guessing.

<!-- COMMONS BEFORE v1 · generated from meta.yaml by tools/notebooks.py · do not edit by hand -->
## Before you start

**Lessons to do first:** `T00-L01`

**This lesson assumes you can already:**

- write a Python class with an __init__ and a few methods, and a function that returns a NamedTuple
- do integer arithmetic with // and %, and say why 7 // 2 is 3 while 7 / 2 is 3.5
- read an HTTP status code such as 429 and say what a request, a response and a header are
- use random.Random(seed).random() to get the same numbers on every run

**New to some of that? These are good places to learn it** (free to read online first, then paid books). You do not need any of them if the list above already reads as familiar.

- [The Python Tutorial, section 9: Classes](https://docs.python.org/3/tutorial/classes.html) · Python Software Foundation · *free* — defining a class with __init__ and methods and keeping state in attributes, which is all TokenBucket and Pacer need
- [The Python Tutorial, section 3: An Informal Introduction to Python (numbers)](https://docs.python.org/3/tutorial/introduction.html) · Python Software Foundation · *free* — / always gives a float while // floors and % keeps the remainder, the whole-number arithmetic the bucket and the planner use instead of floats
- [An overview of HTTP](https://developer.mozilla.org/en-US/docs/Web/HTTP/Guides/Overview) · MDN Web Docs (Mozilla) · *free* — what a request, a response and a header are, which is all the lesson assumes about a 429 and the Retry-After header it may carry
- [random: Generate pseudo-random numbers](https://docs.python.org/3/library/random.html) · Python Software Foundation · *free* — a seeded random.Random gives the same draws on every run, which the class, the start times and the jitter all rely on
- [Exponential Backoff And Jitter](https://aws.amazon.com/blogs/architecture/exponential-backoff-and-jitter/) · Marc Brooker, AWS Architecture Blog · *free* — the post that names full jitter, the retry rule exercise 2 builds, and shows why backing off without it still leaves clumps

In [ ]:
# Setup: everything the lesson needs, in one cell. Standard library only.
import contextlib
import heapq
import io
import json
import math
import os
import random
import signal
import sys
import threading
import time
import traceback
from pathlib import Path
from typing import Callable, NamedTuple, Optional

print("python", sys.version.split()[0])

_FAILED_CHECKS: list[str] = []

# The exercises, in the order you meet them: what the progress board calls each one, and the
# names of the functions whose stubs belong to it. A cell that is waiting on an unfinished
# exercise names it from here.
_EXERCISES: dict[str, tuple[str, tuple[str, ...]]] = {
    "exercise 1": ("TokenBucket", ("__init__", "available", "take", "ready_at_ms")),
    "exercise 2": ("backoff_ms", ("backoff_ms",)),
    "exercise 3": ("Pacer", ("schedule",)),
    "exercise 4": ("class_capacity", ("class_capacity",)),
    "exercise 5": ("pick_next", ("pick_next",)),
}
_STATUS: dict[str, str] = {}   # label -> "passed" | "failed" | "not started", latest run


def _named(labels: list[str]) -> str:
    """["exercise 3"] -> "exercise 3 (Pacer)"; several -> "exercises 3, 4 and 5"."""
    if len(labels) == 1:
        return f"{labels[0]} ({_EXERCISES[labels[0]][0]})"
    nums = [label.split()[-1] for label in labels]
    return "exercises " + ", ".join(nums[:-1]) + " and " + nums[-1]


# How long one check may run before it counts as stuck. A right answer takes milliseconds.
_CHECK_SECONDS = 2
# A demo calls your code thousands of times (a whole class, then ten more classes), so a right
# answer that is merely slow may need longer. It is stopped here, and reported as too slow.
_DEMO_SECONDS = 10


# A time limit for every check that calls the student's code: a loop that never ends must fail
# its check, not freeze the notebook or the grader (which imports the notebook). Copy this block
# unchanged; change only the message. Why it is this careful:
#  * Whoever runs the notebook may already have an alarm running (tools/grade.py stops every
#    rubric row after 45 s; a harness may have one whose default action ends the process). This
#    guard saves it, lets whichever deadline comes first win (handing the signal to the outer
#    handler when it is the outer's turn) and on the way out re-arms it with the time it had left
#    and its interval. Switching it off instead would let the NEXT hang run for ever.
#  * Its own alarm is one-shot, re-armed by hand every 0.25 s once the time is up (in case a bare
#    `except:` swallows the first). A repeating alarm kept firing while a runaway loop's huge list
#    was being torn down; a repeat that arrived after the default handler was back ended the
#    process (exit -14, a dead Jupyter kernel), and one that arrived inside contextlib's exit
#    escaped as a raw timeout and left this handler installed.
#  * It never raises inside its own set-up, exit or clean-up, and on the way out it takes any
#    alarm of its own that has gone off but not yet been handled (with SIGALRM blocked) before
#    the old handler is put back.
# tools/verify_all.py fails a lesson whose guard does less.
class _OutOfTime(BaseException):
    """Raised inside the guarded block when its time is up. A BaseException, so `except
    Exception:` in the code under test cannot swallow it; it repeats every 0.25 s in case a bare
    `except:` does. `_time_limit` turns it into an AssertionError once it has left that code."""


def _resident() -> float:
    """This process's resident memory now, in MiB (Linux: /proc; macOS: libproc); its peak so far elsewhere."""
    try:
        if sys.platform.startswith("linux"):
            with open("/proc/self/statm") as f:
                return int(f.read().split()[1]) * os.sysconf("SC_PAGE_SIZE") / 2**20
        if sys.platform == "darwin":
            import ctypes, struct
            info = ctypes.create_string_buffer(96)           # struct proc_taskinfo
            if ctypes.CDLL("/usr/lib/libproc.dylib").proc_pidinfo(os.getpid(), 4, 0, info, 96) == 96:
                return struct.unpack_from("QQ", info.raw)[1] / 2**20     # pti_resident_size
    except Exception:
        pass
    import resource
    return resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / (2**20 if sys.platform == "darwin" else 2**10)


@contextlib.contextmanager
def _time_limit(seconds: float, what: str, memory_mib: float = 2048):
    """Fail the block with an AssertionError naming `what` if it runs longer than `seconds`.
    Uses the operating system's alarm (Linux and macOS, main thread); a no-op elsewhere."""
    if not (all(hasattr(signal, name) for name in ("setitimer", "SIGALRM", "pthread_sigmask", "sigpending", "sigwait"))
            and threading.current_thread() is threading.main_thread()):
        yield
        return
    if getattr(_time_limit, "stopped", False):           # a stopped block's frames keep what they built in a
        _time_limit.stopped = False                      # reference cycle until now: free it, so it is not counted
        import gc
        gc.collect()                                     # as the growth of this block
    message = (f"{what} was still running after {seconds:g} s: a loop whose stopping condition "
               "is never met runs for ever; give every loop a cap")
    started = time.monotonic()
    used_mib = None
    if memory_mib:
        used_mib = _resident
        base_mib = used_mib()                            # resident memory now; only growth beyond it counts
    previous = signal.getsignal(signal.SIGALRM)
    outer_left, outer_every = signal.getitimer(signal.ITIMER_REAL)   # an alarm already running
    # One alarm at a time, never a repeating one. next: when ours is due; outer: when the alarm we
    # found is due (every: its interval); unseen: an alarm is armed and has not reached _fire yet.
    state = {"next": started + seconds, "outer": started + outer_left if outer_left > 0 else None,
             "every": outer_every, "unseen": False, "fired": False, "done": False, "message": message, "count": 0, "budget": 0,
             "traced": False, "tracer": None, "tool": None, "at": None, "thread": threading.get_ident(),
             "me": sys._getframe(), "owner": None}
    owner = state["me"].f_back
    while owner is not None and owner.f_globals is vars(contextlib):    # contextlib's __enter__
        owner = owner.f_back
    state["owner"], state["at"] = owner, owner.f_lineno   # the frame running the `with`, and its line

    def _arm(now, soonest=0.001):                        # one shot, at whichever deadline is next
        due = [t for t in (state["next"], state["outer"]) if t is not None]
        if used_mib and state["next"] is not None and not state["fired"]:
            due.append(now + 0.1)                        # and look at the memory every 0.1 s
        if due:
            signal.setitimer(signal.ITIMER_REAL, max(min(due) - now, soonest))
            state["unseen"] = True

    def _in_guard(frame):                                # is the alarm landing in a guard's own set-up, exit
        while frame is not None and frame is not state["owner"]:     # or clean-up (this one's or another's:
            code = frame.f_code                          # guards nest), or in contextlib's __exit__ under
            if (frame is state["me"] or code in _own_code or code.co_name in _GUARDS   # a `with`?
                    or frame.f_globals.get("__name__") == "contextlib"):
                return True
            frame = frame.f_back
        return False

    def _fire(signum, frame):
        state["unseen"] = False
        if state["done"]:
            return                                       # a late alarm of ours: already handled
        if _in_guard(frame):                             # never raise inside the guard: it is
            _arm(time.monotonic(), 0.01)                 # finishing; look again shortly
            return
        now = time.monotonic()
        if state["outer"] is not None and now >= state["outer"] - 0.005:
            every = state["every"]                       # the outer alarm is due: act as it would
            state["outer"] = state["outer"] + every * (1 + (now - state["outer"]) // every) if every else None
            try:
                if callable(previous):
                    previous(signum, frame)
                elif previous != signal.SIG_IGN:         # its default action ends the process
                    signal.signal(signal.SIGALRM, signal.SIG_DFL)
                    os.kill(os.getpid(), signal.SIGALRM)
            except BaseException:
                state["next"] = None                     # the outer one is stopping the block
                raise
            finally:
                left, every = signal.getitimer(signal.ITIMER_REAL)
                if left > 0:                             # the outer handler armed its next alarm
                    state["outer"], state["every"] = time.monotonic() + left, every
                _arm(time.monotonic())
            now = time.monotonic()
        grown = used_mib() - base_mib if used_mib and not state["fired"] else 0
        if grown > memory_mib and state["next"] is not None:       # out of memory before out of time
            state["message"] = (f"{what} was using over {memory_mib:g} MiB more memory than when it started, "
                                f"after {now - started:.1f} s: a loop that keeps adding to a list or a string "
                                "never stops; give every loop a cap")
            state["next"] = now
        if state["next"] is not None and now >= state["next"] - 0.005:
            swallowed = state["fired"]                   # a second stop: the first did not end the block
            state["fired"] = True
            state["count"] += 1
            state["next"] = now + 0.25                   # again, in case a bare `except:` swallows it
            _arm(now)
            if swallowed:
                state["budget"] = state["count"] - 1     # the tracer raises this many times, then leaves clean-up alone
                _trace_from(frame)
            raise _OutOfTime(state["message"], state)
        _arm(now)

    def _trace(frame, event, arg):                       # raises the stop at the next line of code that
        if state["done"] or _in_guard(frame):            # runs: inside the handler that swallowed the last
            return None                                  # (the interpreter then unsets it; the next alarm
        if event == "line":                              # sets it again)
            raise _OutOfTime(state["message"], state)
        return _trace

    def _line(code, line):                               # Python 3.12+: sys.monitoring, which is not unset
        if state["done"] or state["budget"] <= 0 or threading.get_ident() != state["thread"]:   # not unset when a
            return                                       # callback raises: the next line the code runs raises too, handlers
        frame = sys._getframe(1)                         # included (this thread only), but only `budget` times:
        if _in_guard(frame) or (frame is state["owner"] and line == state["at"]):   # clean-up that runs once the stop has
            return                                       # left the swallowing loops must not be interrupted again
        state["budget"] -= 1                             # (not the `with` line: its exit runs the guard)
        raise _OutOfTime(state["message"], state)

    def _trace_from(frame):                              # trace the running frames, and the ones called next
        if not state["traced"]:
            state["traced"] = True
            mon = getattr(sys, "monitoring", None)
            for tool in (5, 4, 3, 2, 1, 0) if mon is not None else ():
                try:
                    mon.use_tool_id(tool, "commons-time-limit")
                except ValueError:                       # a debugger or a profiler has this one
                    continue
                state["tool"] = tool
                mon.register_callback(tool, mon.events.LINE, _line)
                mon.set_events(tool, mon.events.LINE)
                return
            state["tracer"] = sys.gettrace()
            sys.settrace(_trace)
        while frame is not None and frame is not state["owner"]:
            frame.f_trace = _trace
            frame = frame.f_back

    _GUARDS = ("_time_limit", "_deadline", "_resident", "_time_limit", "_fire", "_arm", "_trace", "_trace_from", "_line", "_in_guard")
    _own_code = (_fire.__code__, _arm.__code__, _trace.__code__, _trace_from.__code__, _line.__code__)

    signal.signal(signal.SIGALRM, _fire)
    _arm(time.monotonic())
    try:
        try:
            yield
        finally:
            state["done"] = True                         # from here on an alarm of ours only counts
            if state["traced"]:
                state["traced"] = False
                if state["tool"] is not None:
                    sys.monitoring.set_events(state["tool"], 0)
                    sys.monitoring.register_callback(state["tool"], sys.monitoring.events.LINE, None)
                    sys.monitoring.free_tool_id(state["tool"])
                else:
                    sys.settrace(state["tracer"])        # the tracer that was there before, if any
            mask = signal.pthread_sigmask(signal.SIG_BLOCK, {signal.SIGALRM})
            try:
                if signal.setitimer(signal.ITIMER_REAL, 0)[0] == 0 and state["unseen"]:
                    # Ours went off and has not reached _fire: it is pending here, or on its way to
                    # another thread. Take it before the old handler (by default, one that ends the
                    # process) is back.
                    give_up = time.monotonic() + 0.5
                    while state["unseen"] and time.monotonic() < give_up:
                        if signal.SIGALRM in signal.sigpending():
                            signal.sigwait({signal.SIGALRM})
                            state["unseen"] = False
                        else:
                            time.sleep(0.001)            # another thread has it: _fire runs next
                signal.signal(signal.SIGALRM, signal.SIG_DFL if previous is None else previous)
                outer = state["outer"]                   # re-arm the alarm we found, last
                signal.setitimer(signal.ITIMER_REAL, *((max(outer - time.monotonic(), 0.001), state["every"])
                                                       if outer is not None else (0, 0)))
            finally:
                signal.pthread_sigmask(signal.SIG_SETMASK, mask)
                state["me"] = state["owner"] = None
    except _OutOfTime as exc:
        if exc.args[-1] is not state:                    # an enclosing guard's: not ours to report
            raise
        _time_limit.stopped = True
        raise AssertionError(state["message"]) from None
    if state["fired"]:                                   # the block caught our alarm and carried on
        _time_limit.stopped = True
        raise AssertionError(state["message"])


def _try(label: str, check: Callable[[], None], needs: tuple[str, ...] = ()) -> None:
    """Run a check, or a demo that depends on your code, without derailing the notebook.

    A stub you have not filled in yet simply says so. A wrong answer prints the check's own
    message, which names the likely mistake, and the notebook carries on, so one broken
    exercise never hides the feedback on the others. A demo names the exercises it `needs`:
    until each has passed its check, the demo says which one it is waiting for and skips. A
    check still running after `_CHECK_SECONDS` is stopped and counts as failed, so a loop that
    never ends cannot hang the notebook. A demo gets `_DEMO_SECONDS`; one that runs out of time
    is reported as too slow (its exercise has already passed its own check) and is not counted
    as a failed exercise. Nothing is swallowed: every outcome is recorded in `_STATUS` for the
    progress board at the foot of the notebook, and every failed check in `_FAILED_CHECKS`,
    which ends a script run non-zero.
    """
    waiting = [name for name in _EXERCISES   # in the order you meet them
               if name in needs and _STATUS.get(name) != "passed"]
    if waiting:
        _STATUS[label] = "not started"
        print(f"{label}: skipped, needs {_named(waiting)} to pass first.")
        return
    demo = label not in _EXERCISES                       # a demo, not one of the five checks
    try:
        with _time_limit(_DEMO_SECONDS if demo else _CHECK_SECONDS, "this demo" if demo else "this check"):
            check()
    except NotImplementedError as exc:
        _STATUS[label] = "not started"
        stub = traceback.extract_tb(exc.__traceback__)[-1].name   # the frame that raised
        owner = [name for name, (_, frames) in _EXERCISES.items() if stub in frames and name != label]
        if owner:
            print(f"{label}: skipped, needs {_named(owner)} first.")
        elif label in _EXERCISES:
            print(f"{label}: not implemented yet, fill in {_EXERCISES[label][0]} above, then "
                  "re-run this cell.")
        else:
            print(f"{label}: skipped, {stub}() is not implemented yet.")
    except AssertionError as exc:
        _STATUS[label] = "failed"
        if demo and str(exc).startswith("this demo was still running"):
            print(f"{label}: stopped after {_DEMO_SECONDS} s. A loop in your code may never end, or your code may "
                  "be right but too slow for a demo that calls it thousands of times (work out a time by arithmetic "
                  "in ready_at_ms and in schedule, never by stepping a clock forward one millisecond at a time). "
                  "The exercises it needs passed their own checks, so this is not counted as a failed exercise.")
        else:
            _FAILED_CHECKS.append(label)
            print(f"{label}: FAILED, {exc}")
    except Exception as exc:  # a half-finished implementation raising something else
        _STATUS[label] = "failed"
        _FAILED_CHECKS.append(label)
        print(f"{label}: raised {type(exc).__name__}: {exc}")
    else:
        _STATUS[label] = "passed"


def _quick(fn: Callable, *args, what: str):
    """Call your code on a small input with a limit of one second of its own. A small call that takes
    longer is stuck, and a loop that adds to a list as it goes eats gigabytes in seconds, so the
    checks stop it here instead of waiting for the whole check to time out."""
    with _time_limit(1, what):
        return fn(*args)


class _Quick:
    """Wrap an object of yours so that EVERY method call on it gets `_quick`'s one-second limit. The checks
    for exercises 1 and 3 call your methods hundreds of times; a method that loops for ever, or adds to a
    list as it loops, is stopped on the call that sticks, not after the whole check has run out of time."""

    _NOTE = {"ready_at_ms": " (work the wait out by arithmetic; never step a clock forward a millisecond at a time)",
             "schedule": " (ask the buckets when they are ready; never step a clock forward a millisecond at a time)"}

    def __init__(self, obj, what: str) -> None:
        self._obj, self._what = obj, what

    def __getattr__(self, name: str):
        attr = getattr(self._obj, name)
        if not callable(attr):
            return attr
        return lambda *args: _quick(attr, *args, what=f"{self._what}.{name}{self._NOTE.get(name, '')}")


def asset_path(name: str) -> Path:
    """Locate a file that ships in this lesson's assets/ folder. There is no network branch."""
    try:
        here = Path(__file__).resolve().parent
    except NameError:  # a notebook has no __file__
        here = Path.cwd()
    for candidate in (here / "assets" / name, here.parent / "assets" / name,
                      Path.cwd() / "assets" / name):
        if candidate.exists():
            return candidate
    raise FileNotFoundError(f"{name} not found in assets/ beside this lesson; it is never downloaded "
                            "by the lesson code (the launcher cell fetches it when it is missing).")


MS_MIN = 60_000                  # milliseconds in a minute
DAY_MS = 24 * 60 * MS_MIN        # milliseconds in a day: the simulated day starts with the session, at time 0


class Limits(NamedTuple):
    """One tier's published limits. None means the page lists no such limit for this tier."""
    rpm: Optional[int] = None    # requests per minute
    tpm: Optional[int] = None    # tokens per minute
    rpd: Optional[int] = None    # requests per day
    tpd: Optional[int] = None    # tokens per day


SNAPSHOT: dict = json.loads(asset_path("free_tiers.json").read_text())
SNAPSHOT_DATE = SNAPSHOT["snapshot_date"]   # the day each figure was read from the vendor's page
TIERS: dict[str, dict] = {t["id"]: t for t in SNAPSHOT["tiers"]}


def limits_of(tier_id: str) -> Optional[Limits]:
    """The tier's Limits, or None when the vendor publishes no numbers for it."""
    raw = TIERS[tier_id]["limits"]
    return None if raw is None else Limits(**raw)


print(f"free-tier snapshot: {len(TIERS)} tiers, taken {SNAPSHOT_DATE}")

## 1. The snapshot: what the vendors say, on one date

A free tier is a set of limits, each measured over a window: requests per minute (RPM), tokens
per minute (TPM), requests per day (RPD), tokens per day (TPD). Exceed **any one** of them and
the request is refused (Groq, OpenRouter and Cerebras say with HTTP 429, Google with a rate-limit
error). So a tier's real capacity is set by whichever limit you reach first, and which one that
is depends on what you ask for.

Two more things matter before any arithmetic. Each limit has a **scope**. Where a page states
one it names an organisation (Groq, Cerebras), a project (Google, whose page adds that it is not
per API key) or an account (OpenRouter, whose page says more keys or more accounts do not
affect its limits), so a class that shares one account shares one pool. And the figures are
**published summaries**: the pages say the exact limits for an account are shown in its own
console, and one page publishes no free-tier numbers at all. The cell below prints the snapshot.
Read the last two columns: they say which tiers can be planned from the documentation and which
need no purchase or payment method, according to the vendor's page.

In [ ]:
def _cell(value: Optional[int]) -> str:
    return "-" if value is None else f"{value:,}"


def show_snapshot() -> None:
    rows = []
    for tid, tier in TIERS.items():
        lim = limits_of(tid)
        numbers = [_cell(v) for v in (lim or Limits())]
        rows.append((tid, tier["scope"], *numbers,
                     "yes" if lim is not None else "no, unpublished",
                     "yes" if tier["zero_dollar"] else "no"))
    head = ("tier", "scope", "rpm", "tpm", "rpd", "tpd", "plannable", "zero-dollar")
    width = [max(len(str(r[i])) for r in [head, *rows]) for i in range(len(head))]
    for r in [head, *rows]:
        print("  ".join(str(c).rjust(w) if i in (2, 3, 4, 5) else str(c).ljust(w)
                        for i, (c, w) in enumerate(zip(r, width))))
    print()
    for tid, tier in TIERS.items():
        if not tier["zero_dollar"]:
            print(f"  not zero-dollar, {tid}: {tier['why_not_zero_dollar']}")
        if limits_of(tid) is None:
            print(f"  not plannable, {tid}: {tier['note']}")


show_snapshot()
print(f"\nread on {SNAPSHOT_DATE}: a vendor can change any of these without telling this notebook.")

## 2. The class and the notebook

The class is a seeded, fixed set of learners, so every run of this notebook is the same run.
Each learner works through the same **stated notebook**: a loop of independent LLM calls (no
call needs the answer to the one before it). Learners differ in how many calls they need
(some re-run cells), when they press Run, and in their own seeded random draws. Read the
printout; every later number is measured against it.

**How a call is charged.** The model charges every call its prompt plus its maximum completion
and no less: as if each answer used its whole allowance and no part of the prompt were cached.
Cerebras documents estimating a request that way before it runs and then revising the estimate
to the actual usage; Groq says cached tokens do not count towards its limits (`claims.yaml`).
No page read says whether a **daily** limit is charged the estimate or the actual usage, and a
class running one notebook sends the same prompt thirty times. So the token figures below are
the cautious reading, not a prediction.

In [ ]:
SEED = 20261014                  # the authoring date; no other seed was tried
CLASS_SIZE = 30                  # learners behind the quota
SESSION_MS = 90 * MS_MIN         # the lesson slot
PROMPT_TOKENS, MAX_COMPLETION_TOKENS = 1_000, 500
CALL_TOKENS = PROMPT_TOKENS + MAX_COMPLETION_TOKENS   # what every call is charged
BASE_CALLS, MAX_RERUN_CALLS = 12, 6   # calls the notebook needs, and the most any learner re-runs
LATENCY_MS = 3_000               # how long one answer takes to come back
WINDOW_MS = SESSION_MS - LATENCY_MS   # a call must START by then to come back inside the session



def make_class(seed: int) -> list[int]:
    """How many calls each of the class's learners needs, from one seed."""
    rng = random.Random(seed)
    return [BASE_CALLS + int(rng.random() * (MAX_RERUN_CALLS + 1)) for _ in range(CLASS_SIZE)]


DEMANDS = make_class(SEED)
OTHER_SEEDS = tuple(SEED + k for k in range(1, 11))   # ten more classes, for checking: the next ten, not chosen

print(f"{CLASS_SIZE} learners, one {SESSION_MS // MS_MIN}-minute session, seed {SEED} "
      f"(ten more classes, seeds {OTHER_SEEDS[0]} to {OTHER_SEEDS[-1]}, are used to check what follows)")
print(f"a call is charged {CALL_TOKENS:,} tokens ({PROMPT_TOKENS:,} prompt + {MAX_COMPLETION_TOKENS:,} maximum completion)")
print(f"calls needed per learner: {min(DEMANDS)} to {max(DEMANDS)}; the class needs {sum(DEMANDS):,} calls "
      f"and {sum(DEMANDS) * CALL_TOKENS:,} tokens in all")
_published = [lim for lim in (limits_of(t) for t in TIERS) if lim is not None]
print(f"every tier we can plan holds a {CALL_TOKENS:,}-token call in its per-minute token bucket: "
      f"{all(lim.tpm is None or lim.tpm >= CALL_TOKENS for lim in _published)}")

## 3. Exercise 1: `TokenBucket`

A limit of N a minute is not a counter that resets on the minute. Cerebras documents the
model this lesson uses (`claims.yaml` has the sentence): a **token bucket** whose capacity
refills continuously instead of resetting at fixed intervals. The bucket holds at most
`capacity` tokens, starts full, and gains `per_minute / 60,000` of a token every millisecond.
Taking `n` tokens removes them if the bucket holds `n`, and removes nothing if it does not.

Two traps, both about arithmetic. **Fractions:** 7 tokens a minute is one token every 8,571.4
milliseconds, so a bucket that rounds its level each time someone looks at it never refills for
a client that looks every second. **Floats:** a level that is one rounding error short refuses a
call in the very millisecond it should be allowed, and answers one millisecond late when asked
how long to wait; a cell below the check shows both. Keep every amount (tokens, milliseconds,
fractions of a token) as a whole number in a unit you choose.

One rule about the clock. A caller may ask about a time **earlier** than the last one the bucket
saw (the pacer in exercise 3 does: it asks every bucket about the same `not_before_ms`). Read
such a time as the last one the bucket saw: the level never goes backwards, and the answer is
for the latest moment the bucket knows about.

You write four methods. `available` reports whole tokens (fractions rounded down), `take` is
all-or-nothing, and `ready_at_ms` answers the question every client asks next: when? That
answer is **arithmetic**: how much is missing, divided by what one millisecond adds, rounded up.
A loop that steps a clock forward until the bucket holds enough works for a wait of seconds and
not for a wait of a thousand minutes, which the check asks about. Each call to your methods
gets one second.

<details><summary>💡 Hint 1: what to think about</summary>

What should the bucket report after nobody has looked at it for an hour? What should it report
if somebody looks every second for nine seconds? The second answer must not depend on how often
anyone looked. What is the smallest unit in which a token, a millisecond and a fraction of a
token are all whole numbers? And when you ask "when will 3 tokens be there?", should the answer
round down or up, and what happens at the millisecond just before it? What should a look at an
earlier time than the last one report?
</details>
<details><summary>💡 Hint 2: the approach, in words</summary>

Keep the level in sixty-thousandths of a token: one whole token is sixty thousand of them, and
each elapsed millisecond adds `per_minute` of them. Remember the time of the last update, bring
the level forward by the elapsed milliseconds before every look (never backwards), and never let
it pass the capacity. Whole tokens are the level divided by sixty thousand, rounded down. For the
wait, count how many units are missing at the moment you are asked, divide by what one
millisecond adds, and round that up.
</details>

In [ ]:
class TokenBucket:
    """A bucket of at most `capacity` tokens that refills at `per_minute` tokens a minute.

    * It starts full, at `now_ms`.
    * It refills continuously, to the millisecond, and fractions of a token count. At 7 tokens
      a minute one whole token arrives after 8572 ms, not 8571, and looking at the bucket often
      must not change that.
    * Every argument and every result is an integer. A `now_ms` earlier than the last time the
      bucket was used is read as that last time: the level never goes backwards.

    Example:
        >>> b = TokenBucket(10, 60)          # ten tokens, one more each second
        >>> b.take(10, 0)
        True
        >>> b.take(1, 999)                   # too early: refused, and nothing is taken
        False
        >>> b.available(999), b.available(1000)
        (0, 1)
        >>> b.ready_at_ms(3, 1000)           # one token is there; two more take two seconds
        3000
    """

    def __init__(self, capacity: int, per_minute: int, now_ms: int = 0) -> None:
        """Make a full bucket.

        Example:
            >>> TokenBucket(10, 60).available(0)
            10

        Returns: nothing; the bucket is ready to use.
        """
        # YOUR CODE HERE
        raise NotImplementedError

    def available(self, now_ms: int) -> int:
        """Whole tokens in the bucket at `now_ms`, fractions rounded down.

        Example:
            >>> b = TokenBucket(7, 7); b.take(7, 0)
            True
            >>> b.available(8571), b.available(8572)
            (0, 1)

        Returns: an int between 0 and capacity.
        """
        # YOUR CODE HERE
        raise NotImplementedError

    def take(self, n: int, now_ms: int) -> bool:
        """Remove `n` tokens at `now_ms` if the bucket holds at least `n`; otherwise remove none.

        Example:
            >>> b = TokenBucket(5, 60)
            >>> b.take(3, 0), b.take(3, 0), b.available(0)
            (True, False, 2)

        Returns: True if the tokens were taken, False if the bucket was left as it was.
        """
        # YOUR CODE HERE
        raise NotImplementedError

    def ready_at_ms(self, n: int, now_ms: int) -> Optional[int]:
        """The earliest millisecond, `now_ms` or later, at which `take(n, ...)` would succeed.

        Asking changes nothing. `n` more than the capacity can never be taken: return None. The
        answer is arithmetic, not a loop that steps a clock forward: a wait of a thousand minutes
        must come back at once.

        Example:
            >>> b = TokenBucket(7, 7); b.take(7, 0)
            True
            >>> b.ready_at_ms(1, 0)              # 60000 / 7 = 8571.43, so wait for 8572
            8572
            >>> b.ready_at_ms(8, 0) is None
            True

        Returns: an int millisecond, or None when n exceeds the capacity.
        """
        # YOUR CODE HERE
        raise NotImplementedError



def _check_token_bucket() -> None:
    # Every call to your bucket gets a second of its own (_Quick), so a method that never ends is stopped on its call.
    bucket = lambda *args: _Quick(_quick(TokenBucket, *args, what="TokenBucket"), "TokenBucket")
    b = bucket(10, 60)                                # one token a second
    assert b.available(0) == 10, f"a new bucket starts full: expected 10 tokens, got {b.available(0)!r}"
    assert b.take(4, 0) is True and b.available(0) == 6, (
        "taking 4 of 10 should succeed and leave 6; take() returns True/False and removes tokens")
    assert b.take(7, 0) is False and b.available(0) == 6, (
        "taking 7 of 6 must be refused AND must take nothing: a refused take leaves the bucket as it was")
    b = bucket(10, 60)
    assert b.take(10, 0)
    assert b.available(999) == 0 and b.available(1000) == 1, (
        f"at 60 a minute the first token arrives at exactly 1000 ms; got {b.available(999)} at 999 ms and "
        f"{b.available(1000)} at 1000 ms (use >= when you compare the level with what you need)")
    b = bucket(7, 7)
    assert b.take(7, 0)
    assert b.ready_at_ms(1, 0) == 8572, (
        f"7 tokens a minute is one token per 8571.43 ms, so the first whole token is there at 8572 ms; "
        f"got {b.ready_at_ms(1, 0)!r}. Round the wait UP: at 8571 ms the bucket holds 59997 of the 60000 "
        "sixty-thousandths it needs")
    assert b.take(1, 8571) is False and b.take(1, 8572) is True, (
        "one millisecond before ready_at_ms the take must fail and at ready_at_ms it must work")
    b = bucket(7, 7)
    assert b.take(7, 0)
    for t in range(1000, 9001, 1000):                 # somebody looks every second for nine seconds
        seen = b.available(t)
    assert seen == 1, (
        f"looking every second for 9 s at 7 a minute should show 1 token (9000 ms is 1.05 tokens); got "
        f"{seen}. If you round the level each time you look, the fractions are lost: keep the level exact "
        "and round only when you report it")
    assert b.ready_at_ms(8, 9000) is None and b.ready_at_ms(7, 9000) is not None, (
        "n larger than the capacity can never be taken: ready_at_ms returns None for it")
    before = b.available(9000)
    b.ready_at_ms(3, 9000)
    assert b.available(9000) == before, "ready_at_ms only answers; it must not take tokens"
    b = bucket(5, 60)
    assert b.take(5, 0) and b.available(3_600_000) == 5, (
        "an hour of idle time cannot overfill a bucket: the level stops at the capacity")
    b = bucket(1, 6)                                  # one token every 10 s; three looks before it is due
    assert b.take(1, 0)
    for t in (3000, 6000, 9000):
        b.available(t)
    assert b.take(1, 10_000) is True, (
        "6 a minute is one token every 10 s, so after looks at 3, 6 and 9 s the token is there at exactly "
        "10000 ms. A level held in floats can sit one rounding error short here and refuse it: keep whole numbers")
    b = bucket(1, 20)                                 # one token every 3 s; one look after 1 s
    assert b.take(1, 0) and b.available(1000) == 0 and b.ready_at_ms(1, 1000) == 3000, (
        f"20 a minute is one token every 3 s: asked at 1000 ms, the token is due at exactly 3000 ms; got "
        f"{b.ready_at_ms(1, 1000)!r}. A float level makes the wait a hair too long and the ceiling adds a whole "
        "millisecond: keep whole numbers")
    b = bucket(10, 60)                                # a time earlier than the last use is read as the last use
    assert b.take(10, 0) and b.take(1, 3000)          # 3 tokens have arrived by 3000 ms; one is used
    assert b.available(1000) == 2 and b.ready_at_ms(3, 0) == 4000, (
        f"after a take at 3000 ms a look at the earlier time 1000 ms must show what is there at 3000 ms (2 tokens) "
        f"and 3 tokens are due at 4000 ms; got {b.available(1000)} and {b.ready_at_ms(3, 0)!r}. A time earlier "
        "than the bucket's last use is read as that last use: never move the bucket's clock or its level backwards")
    assert b.take(2, 500) is True and b.available(3000) == 0, (
        "a take at the earlier time 500 ms is a take at 3000 ms, where 2 tokens are there")
    b = bucket(1000, 1)                               # a thousand tokens at one a minute: a wait of 1000 minutes
    assert b.take(1000, 0) and b.ready_at_ms(1000, 0) == 60_000_000, (
        "a bucket of 1000 that refills 1 a minute is full again after 1000 minutes, 60000000 ms. ready_at_ms is "
        "arithmetic (how much is missing, divided by what a millisecond adds, rounded up); a loop that steps the "
        "clock one millisecond at a time is far too slow for it")
    print("exercise 1 looks right: a bucket that is exact to the millisecond")

In [ ]:
_try("exercise 1", _check_token_bucket)

Every tier you can plan holds two buckets of this kind, one for requests and one for tokens.
The cell below builds them from the snapshot, empties each, and asks how long it takes to
refill enough for the next call. Whichever bucket takes longer sets the pace of one learner
alone, and `ready_at_ms` is what lets you say so in milliseconds.

In [ ]:
def _bucket_demo() -> None:
    print(f"empty bucket, wait for one call ({CALL_TOKENS:,} tokens, 1 request):")
    for tid in TIERS:
        lim = limits_of(tid)
        if lim is None:
            continue
        waits = {}
        if lim.rpm is not None:
            b = TokenBucket(lim.rpm, lim.rpm)
            b.take(lim.rpm, 0)
            waits["requests"] = (lim.rpm, b.ready_at_ms(1, 0))
        if lim.tpm is not None:
            b = TokenBucket(lim.tpm, lim.tpm)
            b.take(lim.tpm, 0)
            waits["tokens"] = (lim.tpm, b.ready_at_ms(CALL_TOKENS, 0))
        text = ", ".join(f"{kind}: {ms:,} ms after {cap:,} at once" for kind, (cap, ms) in waits.items())
        slower = max(waits, key=lambda k: waits[k][1])
        print(f"  {tid}: {text}; slower, so it sets the pace: {slower}")


_try("bucket demo", _bucket_demo, needs=("exercise 1",))

The float trap, shown. `FloatBucket` is the same bucket with its level held in floating-point
tokens (given code: a warning, not a model to copy). The cell makes it and yours do the same
few things and prints where they part. Floats are exact for round numbers, which is why a float
bucket passes the obvious tests, and inexact for thirds and sixths, which is what a limiter
meets as soon as somebody has looked first.

In [ ]:
class FloatBucket:
    """The same bucket with its level in floating-point tokens."""

    def __init__(self, capacity: int, per_minute: int) -> None:
        self.capacity, self.per_minute, self.level, self.last = capacity, per_minute, float(capacity), 0

    def _advance(self, now_ms: int) -> None:
        if now_ms > self.last:
            self.level = min(float(self.capacity), self.level + (now_ms - self.last) * self.per_minute / MS_MIN)
            self.last = now_ms

    def available(self, now_ms: int) -> int:
        self._advance(now_ms)
        return int(self.level)

    def take(self, n: int, now_ms: int) -> bool:
        self._advance(now_ms)
        if self.level >= n:
            self.level -= n
            return True
        return False

    def ready_at_ms(self, n: int, now_ms: int) -> int:
        self._advance(now_ms)
        return now_ms if self.level >= n else now_ms + math.ceil((n - self.level) * MS_MIN / self.per_minute)


def _float_trap_demo() -> None:
    mine, flt = TokenBucket(1, 6), FloatBucket(1, 6)
    for bucket in (mine, flt):
        bucket.take(1, 0)
        for t in (3000, 6000, 9000):
            bucket.available(t)
    flt.available(10_000)                              # bring it to 10000 ms, so its level can be read
    print("1 token, 6 a minute (one every 10 s), emptied at 0 ms and looked at after 3, 6 and 9 s. At 10000 ms:")
    print(f"  the float bucket holds {flt.level!r} tokens; take(1) works: floats {flt.take(1, 10_000)}, yours {mine.take(1, 10_000)}")
    mine, flt = TokenBucket(1, 20), FloatBucket(1, 20)
    for bucket in (mine, flt):
        bucket.take(1, 0)
        bucket.available(1000)
    print("1 token, 20 a minute (one every 3 s), emptied at 0 ms and looked at once, after 1 s:")
    print(f"  ready_at_ms(1) asked at 1000 ms: floats {flt.ready_at_ms(1, 1000):,} ms, yours {mine.ready_at_ms(1, 1000):,} ms")


_try("float trap demo", _float_trap_demo, needs=("exercise 1",))

## 4. The simulated provider (given code: read it, do not edit it)

`Provider` is one shared account. It checks the two daily limits first, then the request
bucket, then the token bucket, and when it says no it says which limit spoke. For a per-minute
limit it also says when to come back: its `retry_after_ms`, what a `Retry-After` header would
carry. It reports only the **first** limit that failed, as an error response does, so a
client told to wait for the request bucket may find the token bucket empty when it returns.
For a daily limit the wait is the rest of the simulated day. That is how this model resets, a
choice and not a fact: the simulated day starts with the session and the session never crosses
a reset. Real services differ. Google's page says midnight Pacific time, OpenRouter's key
endpoint counts free-model requests per UTC day, and Groq's page states no rule for when a
daily limit resets (`claims.yaml`).

Three modelling choices you should know about. A rejected request consumes nothing: no page read
for this snapshot says whether a real provider charges one, so treat that as unknown, not as
true. `send_retry_after=False` makes the provider omit the hint: OpenRouter's page says an error
response also carries `Retry-After` when every attempted provider returned a hint, and does not
say whether a free-model 429 carries one, though its own platform 429s do carry an
`X-RateLimit-Reset` header, so this models a client that reads only `Retry-After`. And
requests that reach the provider in the **same millisecond** are served in a random order drawn
from the seed, not in learner-number order (`ties="index"` gives that order back). Section 5
measures how much the choice matters.

`run_class` then puts the learners behind one provider. Each learner sends a call, waits for
the answer, and sends the next. A rejected call is retried after the **policy's** delay, and
a learner who has been trying the same call for `PATIENCE_MS` gives up and never finishes
(`patience_ms=None` makes a learner who never gives up).
A policy that reads the hint and is told to come back later than that (a daily limit says
"tomorrow") gives up at once. A policy is just a delay function; exercise 2 writes the rule
this lesson builds from OpenRouter's advice (back off exponentially, honour the header) and
AWS's full jitter.

In [ ]:
class Reply(NamedTuple):
    ok: bool                         # True: the request was served
    retry_after_ms: Optional[int]    # what a Retry-After header would say; None when it sends none
    why: str                         # "ok", or the limit that said no


class Provider:
    """One shared free-tier account. The world your code is measured against."""

    def __init__(self, limits: Limits, send_retry_after: bool = True) -> None:
        self.limits = limits
        self.send_retry_after = send_retry_after
        self.request_bucket = None if limits.rpm is None else TokenBucket(limits.rpm, limits.rpm)
        self.token_bucket = None if limits.tpm is None else TokenBucket(limits.tpm, limits.tpm)
        self.used_requests = 0
        self.used_tokens = 0

    def _no(self, wait_ms: Optional[int], why: str) -> Reply:
        return Reply(False, wait_ms if self.send_retry_after else None, why)

    def request(self, now_ms: int, tokens: int) -> Reply:
        lim = self.limits
        if lim.rpd is not None and self.used_requests >= lim.rpd:
            return self._no(DAY_MS - now_ms, "requests per day")
        if lim.tpd is not None and self.used_tokens + tokens > lim.tpd:
            return self._no(DAY_MS - now_ms, "tokens per day")
        buckets = (("requests per minute", self.request_bucket, 1),
                   ("tokens per minute", self.token_bucket, tokens))
        for why, bucket, need in buckets:
            if bucket is not None:
                ready = bucket.ready_at_ms(need, now_ms)
                if ready is None:
                    return self._no(None, why)      # can never fit: no wait would help
                if ready > now_ms:
                    return self._no(ready - now_ms, why)
        for _, bucket, need in buckets:
            if bucket is not None:
                bucket.take(need, now_ms)
        self.used_requests += 1
        self.used_tokens += tokens
        return Reply(True, None, "ok")


class Outcome(NamedTuple):
    finished: int                    # learners whose last call came back inside the session
    rejected: int                    # requests answered with a 429
    sent: int                        # requests the provider received, served or not
    peak_per_s: int                  # the most requests it received in any one whole second of the clock
    last_finish_ms: Optional[int]    # when the last finisher's last call came back
    by_reason: dict                  # rejections, counted by the limit that said no


class Policy(NamedTuple):
    name: str
    delay: Callable[[int, Optional[int], float], int]   # (retry number from 0, Retry-After, draw) -> ms
    reads_hint: bool                                    # does it look at Retry-After at all?


REJECT_MS = 200                      # how long a 429 takes to come back
START_SPREAD_MS = 30_000             # learners press Run within this long of each other
PATIENCE_MS = 5 * MS_MIN             # how long a learner keeps trying one call
BACKOFF_BASE_MS, BACKOFF_CAP_MS = 1_000, 30_000


def _window(attempt: int) -> int:
    return min(BACKOFF_CAP_MS, BACKOFF_BASE_MS * 2 ** attempt)


def _minutes(ms: Optional[int]) -> str:
    return "-" if ms is None else f"{ms / MS_MIN:.1f} min"


NAIVE = Policy("retry at once", lambda a, hint, draw: 0, reads_hint=False)
OBEY = Policy("wait Retry-After", lambda a, hint, draw: BACKOFF_BASE_MS if hint is None else hint, True)
EXPONENTIAL = Policy("exponential, no jitter", lambda a, hint, draw: max(hint or 0, _window(a)), True)


def run_class(limits: Limits, demands: list[int], policy: Policy, *, shared: bool = True,
              send_retry_after: bool = True, seed: int = SEED, ties: str = "random",
              patience_ms: Optional[int] = PATIENCE_MS) -> Outcome:
    """The class, each learner on its own, in front of `limits`. One account if `shared`,
    otherwise one account per learner. Requests that arrive in the same millisecond are served
    in a random order drawn from the seed (`ties="random"`) or lowest learner number first
    (`ties="index"`). A learner gives up on a call after `patience_ms` of trying it (None: never).
    Deterministic: integer milliseconds and seeded draws."""
    if ties not in ("random", "index"):
        raise ValueError(f"ties must be 'random' or 'index', not {ties!r}")
    n = len(demands)
    providers = [Provider(limits, send_retry_after) for _ in range(1 if shared else n)]
    rngs = [random.Random(seed * 1000 + i) for i in range(n)]
    order = random.Random(seed * 1000 + n)             # a stream of its own for ties; learner i uses seed * 1000 + i
    left, first_try, done, attempt = list(demands), [None] * n, [None] * n, [0] * n
    arrivals: dict[int, int] = {}
    reasons: dict[str, int] = {}
    rejected = sent = 0

    def key() -> float:                                # who goes first among requests with the same millisecond
        return order.random() if ties == "random" else 0.0

    heap = [(int(rngs[i].random() * START_SPREAD_MS), key(), i) for i in range(n)]
    heapq.heapify(heap)
    while heap:
        t, _, i = heapq.heappop(heap)
        if t > WINDOW_MS:
            continue                                   # no time left to start another call
        if first_try[i] is None:
            first_try[i] = t
        reply = providers[0 if shared else i].request(t, CALL_TOKENS)
        sent += 1
        arrivals[t // 1000] = arrivals.get(t // 1000, 0) + 1
        if reply.ok:
            left[i] -= 1
            first_try[i] = None
            attempt[i] = 0
            if left[i] == 0:
                done[i] = t + LATENCY_MS
            else:
                heapq.heappush(heap, (t + LATENCY_MS, key(), i))
            continue
        rejected += 1
        reasons[reply.why] = reasons.get(reply.why, 0) + 1
        attempt[i] += 1
        draw = rngs[i].random()                        # one draw per rejection, whatever the policy
        hint = reply.retry_after_ms if policy.reads_hint else None
        retry_at = t + REJECT_MS + max(0, int(policy.delay(attempt[i] - 1, hint, draw)))
        if patience_ms is None or retry_at - first_try[i] <= patience_ms:
            heapq.heappush(heap, (retry_at, key(), i))   # otherwise this learner gives up for good
    finished = [d for d in done if d is not None]
    return Outcome(len(finished), rejected, sent, max(arrivals.values()),
                   max(finished) if finished else None, reasons)

Before the class, one learner alone. Even then something binds first, and the provider says
which limit it was. The demo needs exercise 1, because the provider is built from your bucket.

In [ ]:
def _provider_demo() -> None:
    lim = limits_of("groq-free-gpt-oss-120b")
    solo = run_class(lim, [DEMANDS[0]], OBEY)
    print(f"one learner alone on groq-free-gpt-oss-120b needs {DEMANDS[0]} calls and finishes: "
          f"{solo.finished == 1}, at {_minutes(solo.last_finish_ms)}")
    print(f"it was told no {solo.rejected} times, by: {dict(sorted(solo.by_reason.items()))}")
    gap = (CALL_TOKENS * MS_MIN) // lim.tpm
    print(f"a call is charged {CALL_TOKENS:,} tokens and the bucket refills {lim.tpm:,} a minute: once the burst "
          f"in a full bucket is spent, calls can start no closer than {gap:,} ms apart on average")


_try("provider demo", _provider_demo, needs=("exercise 1",))

## 5. Exercise 2: `backoff_ms()`

A 429 is not a failure to repeat, it is an instruction to wait. RFC 6585 defines the status and
lets the server add a `Retry-After` header; RFC 9110 says that header gives seconds or a date.
OpenRouter's documentation tells clients to retry with exponential backoff and to honour the
header when it is present (`claims.yaml` quotes both). Two ideas follow. **Exponential**: each
retry waits in a window twice as wide as the last, up to a cap, so a stuck class thins out.
**Jitter**: AWS's account of the same problem says that when all the failed calls back off to
the same time they cause the same contention again, so each learner should pick a random point
in the window rather than its edge. That is "full jitter".

The server's hint is a **minimum**, so the jitter goes **on top of it**: a learner told to wait
twenty seconds who then draws a random point in a one-second window must still wait twenty
seconds and a little more. A rule that takes the larger of the two instead hides a small jitter
under a big hint, and learners rejected in the same instant come back in the same instant. The
demo after the check includes that rule, so you can measure what it costs.

<details><summary>💡 Hint 1: what to think about</summary>

Write down, for retries 0, 1, 2, 3 and 10, how wide the window is when the base is one second
and the cap is thirty. What does the cap limit: the window, the hint, or the sum? What should
happen if the server sent no hint, and is that the same as a hint of zero? What is the largest
and the smallest value the draw can take, and so what range of delays can you return?
</details>
<details><summary>💡 Hint 2: the approach, in words</summary>

Take the smaller of the cap and the base multiplied by two to the power of the retry number:
that is the window. Multiply it by the draw and drop the fraction, so the result is a whole
number of milliseconds somewhere from zero up to just under the window. Then add the server's
hint if there is one. Use the draw exactly once, and return an integer.
</details>

In [ ]:
def backoff_ms(attempt: int, base_ms: int, cap_ms: int, retry_after_ms: Optional[int], draw: float) -> int:
    """Milliseconds to wait before retry number `attempt` (0 for the first retry).

    * window = min(cap_ms, base_ms * 2 ** attempt)
    * jitter = floor(window * draw), where `draw` is in [0, 1): the CALLER draws it from its own
      seeded generator, so runs are reproducible. Use it once.
    * delay = jitter, plus `retry_after_ms` when the server sent one. The cap limits the window,
      never the hint: a server that says wait a minute is obeyed even if `cap_ms` is a second.
      None means no hint; None and 0 both add nothing.

    Example:
        >>> backoff_ms(0, 1000, 30000, None, 0.5)       # window 1000
        500
        >>> backoff_ms(3, 1000, 30000, None, 0.5)       # window 8000
        4000
        >>> backoff_ms(2, 1000, 30000, 5000, 0.25)      # 5000 + floor(4000 * 0.25)
        6000

    Returns: an int number of milliseconds, never negative.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_backoff_ms() -> None:
    backoff = lambda *args: _quick(backoff_ms, *args, what="backoff_ms")   # a call that hangs is stopped at once
    got = [backoff(a, 1000, 30000, None, 0.5) for a in range(7)]
    assert got == [500, 1000, 2000, 4000, 8000, 15000, 15000], (
        f"with base 1000, cap 30000 and a draw of 0.5 the delays should be [500, 1000, 2000, 4000, 8000, "
        f"15000, 15000]; got {got}. The window doubles each retry until the cap (30000) stops it")
    assert backoff(0, 1000, 30000, None, 0.0) == 0, "a draw of 0 is the bottom of the window: delay 0"
    assert backoff(0, 1000, 30000, None, 0.9375) == 937, (
        f"1000 * 0.9375 = 937.5; a delay is a whole number of milliseconds, rounded DOWN: got "
        f"{backoff(0, 1000, 30000, None, 0.9375)!r}")
    assert backoff(2, 1000, 30000, 5000, 0.25) == 6000, (
        f"a 5000 ms Retry-After plus a jitter of floor(4000 * 0.25) = 1000 is 6000; got "
        f"{backoff(2, 1000, 30000, 5000, 0.25)!r}. The jitter goes ON TOP of the hint, not under it")
    assert backoff(0, 1000, 1000, 60000, 0.5) == 60500, (
        "the cap limits the window, not the server's hint: a hint of 60000 with a 1000 cap is 60000 plus jitter")
    assert backoff(1, 1000, 30000, 0, 0.5) == backoff(1, 1000, 30000, None, 0.5), (
        "a hint of 0 adds nothing, exactly like no hint")
    huge = backoff(2000, 1000, 30000, None, 0.5)
    assert huge == 15000 and type(huge) is int, (
        f"retry number 2000 must still give a whole number: the window is the cap (15000 at a draw of 0.5), "
        f"got {huge!r}")
    print("exercise 2 looks right: capped, jittered, and never earlier than the server asked")

In [ ]:
_try("exercise 2", _check_backoff_ms)

The rules below run on one shared account, the same class and the same seed. The tier is OpenRouter's
free variants after a credit purchase, chosen because its quota is enough for everybody in
the long run (the planner in exercise 4 will say so), so what separates the rows is behaviour,
not capacity. The second table repeats the careful rules when the provider sends no
`Retry-After`. Columns: learners finished, requests answered 429, and the highest number of
requests the provider received in any one whole second of the clock.

In [ ]:
HIDDEN_JITTER = Policy("jitter under the hint", lambda a, hint, draw: max(hint or 0, int(_window(a) * draw)), True)
FULL_JITTER = Policy("full jitter on top (yours)",
                     lambda a, hint, draw: backoff_ms(a, BACKOFF_BASE_MS, BACKOFF_CAP_MS, hint, draw), True)
POLICIES = (NAIVE, OBEY, EXPONENTIAL, HIDDEN_JITTER, FULL_JITTER)


def _table(results: list[tuple[Policy, Outcome]]) -> None:
    print(f"  {'rule':<30} {'finished':>8} {'rejected':>10} {'peak per second':>16}")
    for policy, out in results:
        print(f"  {policy.name:<30} {out.finished:>8} {out.rejected:>10,} {out.peak_per_s:>16}")


def _policy_demo() -> None:
    tier = "openrouter-free-variants-10-credits"
    lim = limits_of(tier)
    with_hint = [(p, run_class(lim, DEMANDS, p)) for p in POLICIES]
    print(f"{CLASS_SIZE} learners on one account, {tier}; the provider sends Retry-After:")
    _table(with_hint)
    careful = POLICIES[1:]
    no_hint = [(p, run_class(lim, DEMANDS, p, send_retry_after=False)) for p in careful]
    print("\nthe same, when the provider sends no Retry-After (retrying at once is unchanged):")
    _table(no_hint)
    naive = with_hint[0][1]
    fewest = min(with_hint[1:], key=lambda r: r[1].rejected)
    calm = min(with_hint[1:], key=lambda r: r[1].peak_per_s)
    low, high = min(r[1].finished for r in with_hint[1:]), max(r[1].finished for r in with_hint[1:])
    print(f"\nretrying at once: {naive.rejected:,} rejected requests, {naive.peak_per_s} in its busiest second.")
    print(f"with the hint, in this class: fewest rejected {fewest[0].name} ({fewest[1].rejected:,}); lowest peak "
          f"{calm[0].name} ({calm[1].peak_per_s} a second); the careful rules finished {low} to {high} learners. "
          "The next cell asks which of those columns belong to the rule and which to the model.")


_try("policy demo", _policy_demo, needs=("exercise 1", "exercise 2"))

One class is one sample. The cell below repeats the first table on the other classes (new
demands, new start times, new draws) and counts, for each metric, in how many of all the
classes each rule had the best value. A tie counts for every rule that tied.

In [ ]:
def _policy_seeds_demo() -> None:
    lim = limits_of("openrouter-free-variants-10-credits")
    seeds = (SEED, *OTHER_SEEDS)
    careful = POLICIES[1:]
    metrics = (("fewest rejected", "rejected", min), ("lowest peak per second", "peak_per_s", min),
               ("most finished", "finished", max))
    orders = (("random", "simultaneous requests served in a random order (the default above)"),
              ("index", "simultaneous requests served lowest learner number first"))
    wins = {(ties, metric): {p.name: 0 for p in careful} for ties, _ in orders for metric, _, _ in metrics}
    best_rules = {}                                    # (ties, metric, seed) -> the rules that had the best value
    for seed in seeds:
        demands = make_class(seed)
        for ties, _ in orders:
            runs = [(p, run_class(lim, demands, p, seed=seed, ties=ties)) for p in careful]
            for metric, field, pick in metrics:
                value = pick(getattr(o, field) for _, o in runs)
                best_rules[ties, metric, seed] = {p.name for p, o in runs if getattr(o, field) == value}   # a tie counts for all
                for name in best_rules[ties, metric, seed]:
                    wins[ties, metric][name] += 1
    print(f"classes in which each rule had the best value, of {len(seeds)} (retrying at once is left out):")
    for ties, label in orders:
        print(f"  {label}")
        print(f"    {'':<24}" + "".join(f"{p.name[:26]:>28}" for p in careful))
        for metric, _, _ in metrics:
            print(f"    {metric:<24}" + "".join(f"{wins[ties, metric][p.name]:>28}" for p in careful))
    agree = ", ".join(
        f"{metric} {sum(best_rules['random', metric, seed] == best_rules['index', metric, seed] for seed in seeds)} of {len(seeds)}"
        for metric, _, _ in metrics)
    print(f"  classes in which both orders name the same best rule, by metric: {agree}")


_try("policy seeds", _policy_seeds_demo, needs=("exercise 1", "exercise 2"))

Read the last line first. A column whose best rule survives a change in the order the provider
serves simultaneous requests is telling you about the rule; one that does not is telling you
about the model's tie-break, and a learner starved by a fixed tie order has not "waited
longer", only lost every tie. Waiting longer sends fewer requests but can send its retries in
a clump, and jitter exists to spread retries over time, so judge it on the peak column. The
counts also rest on two more assumptions in the given code (how long a 429 takes to come back,
and how long a learner keeps trying one call): change `REJECT_MS` or `PATIENCE_MS` and they
move, so read the comparison between the rules, not the absolute count. What backoff can never
do is create quota: in this model a daily limit's hint is the rest of the day, so a learner who
reads it stops instead of asking again, and section 8 puts a number on that.

## 6. Exercise 3: `Pacer()`

Backoff reacts to a refusal. A **pacer** avoids it: it knows the limits, so before anything is
sent it can work out the earliest moment at which a request breaks none of them. A teacher who
puts one proxy in front of the class's key puts one pacer there, so the class's requests reach
the provider as one well-spaced stream instead of thirty clumps.

A tier can have four limits at once, and a request must satisfy all of them. A `Pacer` holds a
request bucket (capacity and refill both the requests-per-minute figure), a token bucket (both
the tokens-per-minute figure) and two daily counters. A limit that is `None` is not enforced; a
limit of **0** is, and lets nothing through. `schedule` finds the earliest time a request fits,
**reserves** it (takes from the buckets at that time, adds to the counters) and returns it, or
returns `None` when a daily limit has no room left or a bucket can never hold what the request
needs. Find the time by asking the buckets, not by stepping a clock forward: the demos below
call `schedule` thousands of times.

The trap is order. When two buckets are both short the wait is the **longer** of the two, and
nothing may be taken from either until you know that time. A pacer that takes from the first
bucket and then finds the second empty has spent a request it never sent, and one that says
`None` after taking has spent quota on a refusal.

<details><summary>💡 Hint 1: what to think about</summary>

Which questions can you answer without changing anything? Which of the four limits can only
ever make the answer later, and which can make it never? If the request bucket would be ready
at one moment and the token bucket at another, which moment do you use, and at which moment do
you take from each? What must be true about the pacer after it answers `None`?
</details>
<details><summary>💡 Hint 2: the approach, in words</summary>

First refuse, then wait, then spend. Refuse (return `None`, change nothing) if a daily limit
would be exceeded by this request or a bucket can never hold what it needs. Otherwise ask each
bucket that exists for its ready time, starting from `not_before_ms`; the send time is the
latest of those and `not_before_ms`. Then take from every bucket at that one time, add to both
daily counters, and return it.
</details>

In [ ]:
class Pacer:
    """The earliest-send planner for one shared pool of limits.

    Holds a request bucket and a token bucket (each full at time 0, capacity and refill both
    equal to its per-minute limit, absent when that limit is None) and two daily counters, which
    never reset inside a session. Requests are scheduled in the order they are asked for, and
    `not_before_ms` never goes backwards from one call to the next.
    """

    def __init__(self, limits: Limits) -> None:
        self.limits = limits
        self.requests = None if limits.rpm is None else TokenBucket(limits.rpm, limits.rpm)
        self.tokens = None if limits.tpm is None else TokenBucket(limits.tpm, limits.tpm)
        self.requests_today = 0
        self.tokens_today = 0

    def schedule(self, tokens: int, not_before_ms: int) -> Optional[int]:
        """Reserve the earliest slot, at or after `not_before_ms`, for one request of `tokens`.

        `tokens` is at least 1. The request fits when the request bucket holds 1, the token
        bucket holds `tokens`, and a daily limit (if it has one) has room for it:
        `requests_today + 1 <= rpd` and `tokens_today + tokens <= tpd`. A limit that is None is
        not enforced; a limit of 0 is, and nothing fits it. Returns None, with nothing changed (no
        bucket and no daily counter), when a daily limit has no room or a bucket can never hold what
        the request needs (`tokens` above the token bucket's capacity, or a per-minute limit of 0).
        Otherwise takes 1 request and `tokens` tokens at the returned time and counts them against
        the day.

        Example:
            >>> p = Pacer(Limits(rpm=2, tpm=6000))       # a request every 30 s; 100 tokens a second
            >>> [p.schedule(3000, 0) for _ in range(4)]  # two calls fit at once, then 3000 tokens take 30 s
            [0, 0, 30000, 60000]
            >>> Pacer(Limits(rpd=1)).schedule(10, 0), Pacer(Limits(rpd=0)).schedule(10, 0)
            (0, None)
            >>> Pacer(Limits(rpm=0)).schedule(10, 0) is None      # zero is a limit, not "no limit"
            True

        Returns: the send time in ms, or None.
        """
        # YOUR CODE HERE
        raise NotImplementedError


def _check_pacer() -> None:
    # Every call to schedule gets a second of its own (_Quick), so a pacer that never ends is stopped on its call.
    pacer = lambda limits: _Quick(_quick(Pacer, limits, what="Pacer"), "Pacer")
    p = pacer(Limits(rpm=6))
    got = [p.schedule(100, 0) for _ in range(8)]
    assert got == [0, 0, 0, 0, 0, 0, 10_000, 20_000], (
        f"6 requests a minute is a burst of 6 at time 0, then one every 10 s; got {got}")
    p = pacer(Limits(tpm=3000))
    got = [p.schedule(1000, 0) for _ in range(5)]
    assert got == [0, 0, 0, 20_000, 40_000], (
        f"3000 tokens a minute with 1000-token calls: 3 at once, then one every 20 s; got {got}")
    p = pacer(Limits(rpm=2, tpm=6000))
    got = [p.schedule(3000, 0) for _ in range(4)]
    assert got == [0, 0, 30_000, 60_000], (
        f"both buckets count: two 3000-token calls fit the full buckets at once; then the tokens (3000 in "
        f"30 s) and the requests (one per 30 s) both need 30 s; got {got}. Take from every bucket at the "
        "SAME time, the one you return")
    p = pacer(Limits(rpm=10, tpm=6000))
    got = [p.schedule(t, 0) for t in (1000, 5000, 1000)]
    assert got == [0, 0, 10_000], (
        f"the third call needs 1000 tokens and the bucket holds 0, refilling 100 tokens a second: it "
        f"waits for the TOKEN bucket (10 s), even though the request bucket is nowhere near empty; got {got}")
    p = pacer(Limits(rpm=3, tpm=60_000))
    got = [p.schedule(100, 0) for _ in range(5)]
    assert got == [0, 0, 0, 20_000, 40_000], (
        f"now the REQUEST bucket is the slow one (3 a minute) and the tokens are plentiful; got {got}")
    p = pacer(Limits(rpd=2))
    assert [p.schedule(10, 0) for _ in range(3)] == [0, 0, None], "an rpd of 2 allows two requests, then None"
    p = pacer(Limits(tpd=2500))
    assert [p.schedule(1000, 0) for _ in range(3)] == [0, 0, None], (
        "tpd of 2500 allows two 1000-token calls; the third would make 3000, so it is None")
    assert p.schedule(500, 0) == 0, (
        "a refused request must change nothing: after the None above, a 500-token call (2500 in all) still "
        "fits the day. Refuse before you take or count anything")
    assert pacer(Limits(tpm=1000)).schedule(1001, 0) is None, (
        "a request larger than the token bucket can ever hold is None, not a very late time")
    p = pacer(Limits(tpm=1000, rpd=1))
    got = [p.schedule(1001, 0), p.schedule(10, 0)]
    assert got == [None, 0], (
        f"a call the token bucket can never hold is refused, and a refusal counts nothing against the day: the "
        f"next call, which fits, still has its one request a day; got {got}. Check the buckets BEFORE you count "
        "the call against a daily limit")
    zero = [pacer(Limits(rpm=0)).schedule(10, 0), pacer(Limits(tpm=0)).schedule(1, 0),
            pacer(Limits(rpd=0)).schedule(10, 0), pacer(Limits(tpd=0)).schedule(1, 0)]
    assert zero == [None] * 4, (
        f"a limit of 0 is a limit: nothing may be sent, so the answer is None (rpm, tpm, rpd, tpd gave {zero}). "
        "Only None means 'no limit'; a test written as `if limit:` reads 0 as absent")
    assert pacer(Limits(rpm=60)).schedule(1, 5000) == 5000 and pacer(Limits()).schedule(10**9, 7) == 7, (
        "the answer is never before not_before_ms, and with no limits at all it is not_before_ms")
    print("exercise 3 looks right: one send time that breaks no limit, or None")

In [ ]:
_try("exercise 3", _check_pacer, needs=("exercise 1",))

`run_proxy` puts the class behind one proxy that owns the key: the proxy asks its pacer for a
send time, sends the request to the provider at that time, and asks a `pick` function whose call
goes next. A request the pacer thought was fine can still be refused (the provider is a
separate piece of code, and `believed` lets the pacer hold different limits from the
provider's), so the proxy counts those and waits as told. Learners' calls are independent, so
the proxy may have several of one learner's calls in flight. Learners behind the proxy wait as
long as it takes: unlike the learners of `run_class`, nobody gives up after `PATIENCE_MS`. And
a pacer reserves quota before the provider answers, so a refused request still counts against
the pacer's own day (the provider, in this model, charges nothing for a refusal).

In [ ]:
def run_proxy(limits: Limits, demands: list[int], pick: Callable[[list[int], Optional[int]], Optional[int]],
              *, believed: Optional[Limits] = None, first_sent: Optional[list] = None) -> Outcome:
    """The class behind one proxy. `pick(remaining, last)` returns the index of the learner whose call
    goes next (remaining[i] is learner i's calls still to make; last is who went last, or None) or None
    for nobody. The pacer is built from `believed` when given, else from the real `limits`. No learner
    gives up. If `first_sent` is a list it is filled with the time each learner's first call was sent
    (None for a learner whose call never went)."""
    n = len(demands)
    provider, pacer = Provider(limits), Pacer(limits if believed is None else believed)
    left, done = list(demands), [None] * n
    arrivals: dict[int, int] = {}
    reasons: dict[str, int] = {}
    rejected = sent = 0
    now, last = 0, None
    if first_sent is not None:
        first_sent[:] = [None] * n
    while any(c > 0 for c in left):
        who = pick(list(left), last)
        if who is None:
            break
        assert left[who] > 0, f"pick chose learner {who}, who has no calls left"
        t = pacer.schedule(CALL_TOKENS, now)
        if t is None or t > WINDOW_MS:
            break                                     # the day's quota, or the session, is over
        now = max(now, t)
        reply = provider.request(now, CALL_TOKENS)
        sent += 1
        arrivals[now // 1000] = arrivals.get(now // 1000, 0) + 1
        if first_sent is not None and first_sent[who] is None:
            first_sent[who] = now
        if not reply.ok:
            rejected += 1
            reasons[reply.why] = reasons.get(reply.why, 0) + 1
            now += reply.retry_after_ms or 1_000      # wait as told, or a second if told nothing
            continue
        left[who] -= 1
        last = who
        if left[who] == 0:
            done[who] = now + LATENCY_MS
    finished = [d for d in done if d is not None]
    return Outcome(len(finished), rejected, sent, max(arrivals.values(), default=0),
                   max(finished) if finished else None, reasons)


def first_come(remaining: list[int], last: Optional[int]) -> Optional[int]:
    """The simplest order: the lowest-numbered learner who still has calls."""
    return next((i for i, c in enumerate(remaining) if c > 0), None)


def _pacer_demo() -> None:
    tier = "openrouter-free-variants-10-credits"
    lim = limits_of(tier)
    patient = run_class(lim, DEMANDS, EXPONENTIAL, patience_ms=None)
    five_min = run_class(lim, DEMANDS, EXPONENTIAL)
    first_sent: list = []
    proxied = run_proxy(lim, DEMANDS, first_come, first_sent=first_sent)
    print(f"the same class on {tier}:")
    print(f"  {'':<44} {'finished':>8} {'rejected':>9} {'sent':>7} {'last answer':>12}")
    rows = ((f"each learner alone, gives up after {PATIENCE_MS // MS_MIN} min", five_min),
            ("each learner alone, never gives up", patient),
            ("behind one proxy with a pacer, nobody gives up", proxied))
    for name, o in rows:
        print(f"  {name:<44} {o.finished:>8} {o.rejected:>9,} {o.sent:>7,} {_minutes(o.last_finish_ms):>12}")
    agree = "agree" if proxied.rejected == 0 else "do not fully agree"
    print(f"the proxy's rejected count is a measurement: it asked the provider {proxied.sent:,} times and was "
          f"refused {proxied.rejected:,} times, so its pacer and the provider {agree} about the limits.")
    waits = [t for t in first_sent if t is not None]
    late = sum(t > PATIENCE_MS for t in waits)
    print(f"the finished column compares patience, not pacing: with no pacer at all, never giving up takes the class "
          f"from {five_min.finished} to {patient.finished} finishers. Pacing shows in the rejected column "
          f"({patient.rejected:,} when nobody gives up, against {proxied.rejected:,} behind the proxy) and costs "
          f"waiting: the last learner's first call goes out {_minutes(max(waits))} in, and {late} of {CLASS_SIZE} "
          f"first calls go out later than the {PATIENCE_MS // MS_MIN} minutes the uncoordinated learners would keep "
          "trying one call.")


_try("pacer demo", _pacer_demo, needs=("exercise 1", "exercise 3"))

## 7. Exercise 4: `class_capacity()`

So far you have been running classes. A teacher needs the answer before the session starts.
Each limit, alone, supplies only so many calls inside the session. A bucket that starts full
and refills continuously supplies its capacity plus whatever has refilled by the time the last
call must start, `window_ms` (a call that starts later does not come back inside the session).
That is `rpm * (60,000 + window) / 60,000` calls for the request limit, the same for tokens
divided by the tokens in a call, and the daily limits give their figure straight (daily tokens
divided by the call size). Always multiply before you divide, so nothing is rounded early.

With a budget of calls the question is how many **learners** can finish. Choose the learners who
need the **fewest** calls first, because every call spent on a learner who cannot finish is
wasted. A tier's capacity is the smallest answer over its limits, and the limit that gives it
is the one that **binds**. That name is what a teacher acts on: it says whether to ask for
more per-minute headroom, or to split the class, or to pick another tier.

<details><summary>💡 Hint 1: what to think about</summary>

Treat each limit as its own question: how many calls could this limit alone supply in the
window? Which of the four need the size of a call, and which do not? Given a budget of calls and
a list of demands, which learners should you pick to finish as many as possible, and when do
you stop? What do you report when two limits give the same answer, and when no limit stops
anyone?
</details>
<details><summary>💡 Hint 2: the approach, in words</summary>

For each limit that is set, work out the calls it supplies using integer arithmetic only. Sort
the demands and add them smallest first for as long as the running total stays within that
budget; the number added is that limit's learner count. The answer is the smallest of those
counts. Name the first limit, in the order requests per minute, tokens per minute, requests per
day, tokens per day, whose count equals it. If the answer is the whole class, nothing binds.
</details>

In [ ]:
LIMIT_ORDER = ("rpm", "tpm", "rpd", "tpd")


class Capacity(NamedTuple):
    finishers: int                  # at most this many learners can finish
    binding: Optional[str]          # the limit that holds it there, or None when nothing does
    by_limit: dict                  # for each limit that is set: how many learners it alone allows


def class_capacity(limits: Limits, demands: list[int], tokens_per_call: int, window_ms: int) -> Capacity:
    """An upper bound on finishers when the whole class shares ONE pool of `limits`.

    A limit that is None is not counted. A limit of 0 is set: it supplies no calls, so nobody
    finishes and it binds. For each limit that is set, the calls it supplies inside the window
    (integer arithmetic, floor division, multiply before dividing):

    * rpm: rpm * (60000 + window_ms) // 60000
    * tpm: tpm * (60000 + window_ms) // (60000 * tokens_per_call)
    * rpd: rpd
    * tpd: tpd // tokens_per_call

    The learners that limit alone lets finish: the largest k such that the k SMALLEST demands
    sum to at most the calls it supplies (a sum equal to the budget fits). `finishers` is the
    smallest of those counts, or len(demands) when no limit is set. `binding` is the first of
    "rpm", "tpm", "rpd", "tpd" whose count equals `finishers`, or None when `finishers` is
    len(demands): nothing stops anyone.

    Example:
        >>> class_capacity(Limits(rpd=10), [4, 3, 5, 2], 100, 3_600_000)      # 2 + 3 + 4 = 9 <= 10
        Capacity(finishers=3, binding='rpd', by_limit={'rpd': 3})
        >>> class_capacity(Limits(), [4, 3], 100, 0)
        Capacity(finishers=2, binding=None, by_limit={})

    Returns: a Capacity.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_class_capacity() -> None:
    plan = lambda *args: _quick(class_capacity, *args, what="class_capacity")   # a hang is stopped at once
    got = plan(Limits(rpd=10), [4, 3, 5, 2], 100, 3_600_000)
    assert got == Capacity(3, "rpd", {"rpd": 3}), (
        f"10 calls a day and demands 2, 3, 4, 5: the three smallest need 9 and fit, adding the 5 makes 14; "
        f"expected Capacity(3, 'rpd', {{'rpd': 3}}), got {got!r}. Take the SMALLEST demands first, not the "
        "first ones in the list")
    got = plan(Limits(rpd=9), [4, 3, 5, 2], 100, 0)
    assert got.finishers == 3, (
        f"a running total exactly equal to the budget fits (2 + 3 + 4 = 9 <= 9); got {got.finishers}")
    got = plan(Limits(rpm=2), [3] * 5, 100, 60_000)
    assert got.by_limit == {"rpm": 1} and got.binding == "rpm", (
        f"2 a minute starts full and refills 2 more in a minute: 4 calls in all, so one learner of 3 calls "
        f"finishes and two would need 6; got {got!r}. Count the full bucket as well as the refill")
    got = plan(Limits(tpm=3000), [2, 2, 2], 1000, 60_000)
    assert got.by_limit == {"tpm": 3} and got.finishers == 3 and got.binding is None, (
        f"3000 tokens a minute with 1000-token calls over a minute supplies 6 calls, enough for all three "
        f"learners of 2 calls: nothing binds; got {got!r}")
    got = plan(Limits(tpm=3000, tpd=3000), [1, 1, 1], 1000, 0)
    assert got.by_limit == {"tpm": 3, "tpd": 3} and got.finishers == 3, (
        f"tpm 3000 over a zero-length window is 3 calls; tpd 3000 is 3 calls; got {got!r}")
    got = plan(Limits(rpm=1, tpm=1000, rpd=3, tpd=2000), [1, 1, 1, 1], 1000, 0)
    assert got.by_limit == {"rpm": 1, "tpm": 1, "rpd": 3, "tpd": 2} and got.finishers == 1 and got.binding == "rpm", (
        f"counts rpm 1, tpm 1, rpd 3, tpd 2: the answer is 1 and the first limit in the order rpm, tpm, rpd, "
        f"tpd that gives 1 is rpm; got {got!r}")
    assert plan(Limits(), [4, 3], 100, 0) == Capacity(2, None, {}), (
        "with no limit set everyone can finish and nothing binds")
    for limits, want in ((Limits(rpm=0, rpd=10), Capacity(0, "rpm", {"rpm": 0, "rpd": 2})),
                         (Limits(rpm=5, rpd=0), Capacity(0, "rpd", {"rpm": 2, "rpd": 0}))):
        got = plan(limits, [1, 1], 100, 60_000)
        assert got == want, (
            f"{limits}: a limit of 0 supplies no calls, so nobody finishes and that limit binds; expected {want!r}, "
            f"got {got!r}. Only None means 'no limit'; a test written as `if limit:` reads 0 as absent")
    print("exercise 4 looks right: how many learners fit, and which limit says so")

In [ ]:
_try("exercise 4", _check_class_capacity)

The planner, on every tier whose limits the vendor publishes, for the class and the notebook
from section 2. Each limit column is how many learners **that limit alone** allows; the answer
is the smallest of them, and the last column names the limit that gives it. A `-` is a limit
the page does not list. A tier with no published figures is not planned at all.

In [ ]:
def _capacity_demo() -> None:
    print(f"{CLASS_SIZE} learners, {sum(DEMANDS):,} calls needed, a {SESSION_MS // MS_MIN}-minute session; "
          "each limit column is the learners that limit alone allows:")
    print(f"  {'tier':<38} " + " ".join(f"{n:>5}" for n in LIMIT_ORDER) + f" {'finishers':>10}  binding")
    for tid in TIERS:
        lim = limits_of(tid)
        if lim is None:
            print(f"  {tid:<38} not planned: the vendor publishes no figures for it")
            continue
        cap = class_capacity(lim, DEMANDS, CALL_TOKENS, WINDOW_MS)
        cols = " ".join(f"{cap.by_limit.get(n, '-'):>5}" for n in LIMIT_ORDER)
        print(f"  {tid:<38} {cols} {cap.finishers:>10}  {cap.binding or 'none'}")
    free = [t for t in TIERS if TIERS[t]["zero_dollar"] and limits_of(t) is not None]
    short = [t for t in free if class_capacity(limits_of(t), DEMANDS, CALL_TOKENS, WINDOW_MS).finishers < CLASS_SIZE]
    print(f"\nplannable and zero-dollar: {', '.join(free)}.")
    print(f"of those, a whole class on one account cannot all finish on: {', '.join(short) if short else 'none'}.")


_try("capacity demo", _capacity_demo, needs=("exercise 4",))

## 8. Exercise 5: `pick_next()`

Same quota, same learners, same pacer. The only choice left is **whose call goes next**, and a
learner with half a notebook done has not finished anything. Three orders:

- `round_robin`: everyone in turn, starting after whoever went last. It looks fair.
- `fifo`: the lowest-numbered learner with calls left.
- `fewest_left`: the learner with the fewest calls left (ties to the lower number).

`remaining[i]` is how many calls learner `i` still has to make; 0 means finished. `last` is
who went last, or `None` at the start. Return the index of the next learner, or `None` when
nobody has calls left. A policy name that is not one of the three raises `ValueError`, even
when nobody has calls left.

<details><summary>💡 Hint 1: what to think about</summary>

What does each name promise to do when learner 0 has just gone? What should the round robin do
at the end of the list, and what should it do about a learner who has finished? When two
learners are tied for fewest calls left, who goes first, and why must the answer be the same
every time? What comes back when nobody has calls left?
</details>
<details><summary>💡 Hint 2: the approach, in words</summary>

First check the policy name. Then make the list of learners who still have calls left. `fifo` is
the first of them. `fewest_left` is the one with the smallest count, breaking ties by the lower
index. `round_robin` starts looking at the learner after `last` (at 0 when there is no last)
and walks round the whole list, wrapping at the end, until it meets a learner with calls left.
</details>

In [ ]:
POLICY_NAMES = ("round_robin", "fifo", "fewest_left")


def pick_next(policy: str, remaining: list[int], last: Optional[int]) -> Optional[int]:
    """The index of the learner whose call goes next under `policy`, or None if nobody has calls left.

    * `fifo`: the lowest index with remaining > 0.
    * `fewest_left`: the index with the smallest remaining > 0; ties go to the lower index.
    * `round_robin`: the first index after `last` (from 0 when `last` is None), wrapping past the
      end, with remaining > 0. `last` itself is tried last, and only if it still has calls left.
    * any other policy name: ValueError, even when nobody has calls left.

    Example:
        >>> pick_next("fifo", [0, 2, 3], None), pick_next("fewest_left", [3, 2, 2], 0)
        (1, 1)
        >>> pick_next("round_robin", [1, 1, 1], 2), pick_next("round_robin", [0, 2, 3], 2)
        (0, 1)

    Returns: an int index, or None.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_pick_next() -> None:
    pick = lambda *args: _quick(pick_next, *args, what="pick_next")   # a hang is stopped after 1 s, before it fills the memory
    assert pick("fifo", [0, 2, 3], None) == 1 and pick("fifo", [3, 0, 2], 2) == 0, (
        "fifo skips finished learners: the lowest index with calls left, whoever went last")
    assert pick("fewest_left", [3, 2, 5], None) == 1 and pick("fewest_left", [3, 2, 2], 0) == 1, (
        "fewest_left takes the smallest remaining count above zero, and the LOWER index on a tie")
    assert pick("fewest_left", [0, 4, 9], None) == 1, "a finished learner (0 left) is never 'fewest'"
    got = [pick("round_robin", [2, 2, 2], last) for last in (None, 0, 1, 2)]
    assert got == [0, 1, 2, 0], (
        f"round robin goes to the next learner after the last, and wraps at the end; got {got} for last = "
        "None, 0, 1, 2")
    assert pick("round_robin", [0, 2, 3], 2) == 1 and pick("round_robin", [3, 0, 0], 0) == 0, (
        "round robin skips finished learners as it wraps, and the last learner is tried last: if nobody else "
        "has calls left it goes again")
    for policy in POLICY_NAMES:
        assert pick(policy, [0, 0, 0], 1) is None and pick(policy, [], None) is None, (
            f"{policy}: when nobody has calls left the answer is None, not an index")
    for args in (("lifo", [1, 2], None), ("lifo", [0, 0], None)):
        try:
            pick(*args)
        except ValueError:
            continue
        raise AssertionError(f"an unknown policy name must raise ValueError, even for {args[1]}")
    print("exercise 5 looks right: three orders, one function")

In [ ]:
_try("exercise 5", _check_pick_next)

Now the same class, the same quota and the same pacer, served in each order, beside a class of
uncoordinated learners and beside a class where every learner has an account of their own. That
last row is thirty people each signing up for themselves. Check each vendor's terms before you
plan on that: OpenRouter's forbid one user creating several accounts to get round its limits
(`claims.yaml` quotes the clause), which is not thirty users with one account each, and its page
says extra accounts or keys do not raise its limits anyway. The lesson did not read Groq's
terms and says nothing about them. The planner's number from exercise 4 is printed first: it
is an **upper bound**, so a run can match it or fall short, never beat it. The tiers are the
ones that cost nothing.

In [ ]:
def _order_demo() -> None:
    for tid in (t for t in TIERS if TIERS[t]["zero_dollar"] and limits_of(t) is not None):
        lim = limits_of(tid)
        cap = class_capacity(lim, DEMANDS, CALL_TOKENS, WINDOW_MS)
        print(f"{tid}: the planner says at most {cap.finishers} of {CLASS_SIZE} finish, set by {cap.binding or 'nothing'}")
        runs = [("uncoordinated, retry at once", run_class(lim, DEMANDS, NAIVE)),
                ("uncoordinated, full jitter (yours)", run_class(lim, DEMANDS, FULL_JITTER))]
        for name in POLICY_NAMES:
            runs.append((f"one proxy, {name}", run_proxy(lim, DEMANDS, lambda r, l, name=name: pick_next(name, r, l))))
        runs.append((f"{CLASS_SIZE} learners, an account each, full jitter", run_class(lim, DEMANDS, FULL_JITTER, shared=False)))
        print(f"  {'':<38} {'finished':>8} {'rejected':>10} {'last answer':>12}")
        for name, o in runs:
            print(f"  {name:<38} {o.finished:>8} {o.rejected:>10,} {_minutes(o.last_finish_ms):>12}")
        reached = [n for n, o in runs[2:5] if o.finished == cap.finishers]
        beat = [n for n, o in runs[:5] if o.finished > cap.finishers]
        print(f"  reached the planner's number: {', '.join(reached) if reached else 'none of the three orders'}; "
              f"exceeded it: {', '.join(beat) if beat else 'nothing'}\n")


_try("order demo", _order_demo, needs=("exercise 1", "exercise 2", "exercise 3", "exercise 4", "exercise 5"))

Again, one class is one sample. For each zero-dollar tier the cell below serves all the classes
in each order and counts how often the order reached the planner's number for that class, and
the fewest and most learners it finished.

In [ ]:
def _order_seeds_demo() -> None:
    seeds = (SEED, *OTHER_SEEDS)
    for tid in (t for t in TIERS if TIERS[t]["zero_dollar"] and limits_of(t) is not None):
        lim = limits_of(tid)
        reached = {name: 0 for name in POLICY_NAMES}
        spread = {name: [CLASS_SIZE, 0] for name in POLICY_NAMES}
        bounds = []
        for seed in seeds:
            demands = make_class(seed)
            bound = class_capacity(lim, demands, CALL_TOKENS, WINDOW_MS).finishers
            bounds.append(bound)
            for name in POLICY_NAMES:
                o = run_proxy(lim, demands, lambda r, l, name=name: pick_next(name, r, l))
                reached[name] += o.finished == bound
                assert o.finished <= bound, f"{name} finished {o.finished} where the planner's bound is {bound}"
                spread[name] = [min(spread[name][0], o.finished), max(spread[name][1], o.finished)]
        print(f"{tid}: planner's number over the {len(seeds)} classes runs from {min(bounds)} to {max(bounds)}")
        for name in POLICY_NAMES:
            low, high = spread[name]
            print(f"  {name:<12} reached it in {reached[name]:>2} of {len(seeds)} classes; finished {low} to {high}")


_try("order seeds", _order_seeds_demo, needs=("exercise 1", "exercise 3", "exercise 4", "exercise 5"))

Read the rows for the order a shared key serves in. Round robin spends the quota evenly, so when
it will not stretch to the whole class it can leave every learner part-way; serving learners to
completion, the one with the fewest calls left first, spends the same quota on notebooks that
finish. The planner's number is the most any order can reach, and the line under each table
says which orders did. For the uncoordinated rows compare the rejected column: a rule that
reads a daily limit's hint stops asking, and one that does not keeps asking until its patience
is gone.

## 9. When one provider cuts its quota

This is a **what-if**, not history: the lesson does not claim that any vendor has made these
cuts. Each case takes one tier from the snapshot and cuts one limit in memory. The planner is
re-run on the cut limits, and so is the proxy, twice: once with a pacer that still believes the
snapshot, and once with a pacer told the new figures. The pacer's belief is where a stale
snapshot shows up: a stale daily figure promises the class something that is not there, and a
stale per-minute figure sends requests that are refused.

In [ ]:
def scaled(limits: Limits, field: str, percent: int) -> Limits:
    """`limits` with one field cut to `percent` of itself (whole numbers, rounded down)."""
    return limits._replace(**{field: getattr(limits, field) * percent // 100})


def _fewest_left(remaining: list[int], last: Optional[int]) -> Optional[int]:
    return pick_next("fewest_left", remaining, last)


def _cut_demo() -> None:
    cases = (("groq-free-gpt-oss-120b", "tpd", 50), ("openrouter-free-variants-10-credits", "rpm", 50),
             ("openrouter-free-variants-10-credits", "rpd", 25), ("groq-free-gpt-oss-120b", "rpm", 0))
    for tid, field, percent in cases:
        old = limits_of(tid)
        new = scaled(old, field, percent)
        before = class_capacity(old, DEMANDS, CALL_TOKENS, WINDOW_MS)
        after = class_capacity(new, DEMANDS, CALL_TOKENS, WINDOW_MS)
        stale = run_proxy(new, DEMANDS, _fewest_left, believed=old)
        told = run_proxy(new, DEMANDS, _fewest_left)
        print(f"what if {tid} cut {field} from {getattr(old, field):,} to {getattr(new, field):,}:")
        print(f"  planner, old figures: {before.finishers} finish, set by {before.binding or 'nothing'}; "
              f"cut figures: {after.finishers} finish, set by {after.binding or 'nothing'}")
        print(f"  proxy whose pacer trusts the old snapshot: {stale.finished} finished, refused {stale.rejected:,}")
        print(f"  proxy whose pacer was told the cut:        {told.finished} finished, refused {told.rejected:,}")
        if told.sent == 0 and stale.sent and old.tpd is not None:
            print(f"  the stale pacer stopped after {stale.sent:,} requests; its believed daily tokens hold "
                  f"{old.tpd // CALL_TOKENS:,} calls. A pacer charges a request to its own day when it schedules it, "
                  "refused or not, so it used up a day that the provider never charged")
        broke = before.finishers - told.finished
        print(f"  a class promised {before.finishers} on the old figures got {told.finished}"
              + (f": {broke} learners short" if broke > 0 else ": the promise survived") + "\n")


_try("cut demo", _cut_demo, needs=("exercise 1", "exercise 3", "exercise 4", "exercise 5"))

Compare the cases above. A cut to a per-minute figure changes how fast calls can be sent; a
cut to a daily figure changes how many can be sent at all, and a pacer holding the old daily
figure learns of it only when the provider refuses. The last case cuts a limit to 0, which is
a limit and not an absence: told, the pacer sends nothing; not told, it keeps asking a provider
that can only say no, until its own daily counters (which charge refused requests as well) run
out or the session ends. None of this is visible without a dated snapshot to compare against,
which is why the snapshot carries its date and why the planner should be re-run whenever the
snapshot is.

## 10. Common mistakes

- **Retrying a 429 at once.** The refusal is an instruction to wait. The table of rules above
  is the measurement: count the rejected requests of the first rule.
- **Putting the jitter under the hint** (taking the larger of the two). Learners rejected in the
  same instant are told the same wait and come back together. Add the jitter on top.
- **Retrying a daily limit.** In this model the hint is the rest of the simulated day, so no
  retry inside the lesson can succeed: stop, and tell the learner which limit spoke. Real
  services reset in their own ways (section 4), so read the hint instead of assuming it.
- **Assuming each key has its own allowance.** Where a page states a scope it is an
  organisation, a project or an account, not a key; OpenRouter's adds that more keys or accounts
  do not raise its limits. One key for the class is one pool. Learners who each sign up for an
  account of their own have one pool each, a different thing; whether that is allowed is for
  each vendor's terms to say, and OpenRouter's forbid one person making several accounts.
- **Treating an unpublished limit as unlimited.** The Gemini row has no figures, so the planner
  refuses it. Read the number off the console and put it in a dated snapshot of your own.
- **Serving a short quota fairly.** When the quota will not stretch to the whole class, fair
  round robin can leave everybody part-way. Serve to completion.
- **Floats in a limiter.** A level one rounding error short refuses a request in the very
  millisecond it should be allowed, and two ways of writing the sum can disagree.
- **Believing the simulation about a provider.** It models documented rules. Whether a real
  service charges a refused request, or counts in windows rather than buckets, only testing
  it will tell you.

One of those is worth checking rather than believing: the tier the planner cannot plan.

In [ ]:
def _unknown_is_not_unlimited() -> None:
    for tid, tier in TIERS.items():
        if limits_of(tid) is None:
            print(f"{tid}: limits are None, so there is no Limits to hand to a planner. Its page says:")
            print(f"  {tier['note']}")
            print(f"what a planner given Limits() (no limit set) would say for it: "
                  f"{class_capacity(Limits(), DEMANDS, CALL_TOKENS, WINDOW_MS).finishers} of {CLASS_SIZE} finish, "
                  "which is the planner believing unlimited, not knowing.")


_try("unknown is not unlimited", _unknown_is_not_unlimited, needs=("exercise 4",))

## 11. Self-check

1. Thirty learners each create a key inside one organisation on a free tier whose limits are
   scoped to the organisation. The class's allowance is:
   - (a) thirty times the published figure, one per key
   - (b) the published figure, shared by every key in the organisation
   - (c) the published figure divided by thirty, enforced for each learner

2. A request comes back 429 with a `Retry-After` many hours long. The right response is:
   - (a) stop for today and report which limit was reached
   - (b) retry at once, because a refused request costs nothing
   - (c) retry in a few seconds with jitter, in case the hint is wrong

3. A client takes the larger of the server's `Retry-After` and its own jitter, so learners
   rejected together all wait the hint. What went wrong?
   - (a) the hint should have been ignored
   - (b) the jitter should have been drawn before the request was sent
   - (c) the jitter should have been added on top of the hint, so that equal hints still spread

4. The daily quota covers about two thirds of a class's calls. Which order of service lets
   the most learners finish a notebook?
   - (a) round robin, because it is fair
   - (b) fewest calls left first, because quota spent on a learner who finishes counts and
         quota spread over learners who do not is lost
   - (c) first come first served, because order of arrival is neutral

5. `class_capacity` says at most 10 learners can finish, and a run behind a proxy finishes
   8. The most likely reading is:
   - (a) the bound is an upper bound; the order of service, the pacer or the provider kept
         the run below it, and the gap is the thing to investigate
   - (b) the planner is wrong, because a bound must be reached
   - (c) the provider cut its quota during the run

Answers come with this lesson's worked solution when you enrol on Synapsa.

## What you built, and where it goes next

You built the arithmetic a teacher needs before a class meets a free tier: a limiter exact to
the millisecond, a retry rule that waits as told and spreads out, a pacer that never asks too
early, a planner that says how many learners fit and what stops the rest, and an order of
service that turns a short quota into finished notebooks. The route to a live class is to
replace `Provider` with real calls, keep the proxy, and re-take `assets/free_tiers.json` from
the vendors' pages each term. Automatic failover when one free quota is exhausted would be the
same planner run on a second tier, sending each learner where quota remains. OpenRouter's page
also says its limits differ for different models and suggests sharing the load that way; it does
not say whether the 20 requests a minute are one pool for every `:free` model, and that decides
whether doing so helps.

In [ ]:
_MARKS = {"passed": "✅", "failed": "❌", "not started": "⏳"}


def _progress_board() -> None:
    """One line per exercise, from the latest run of its check, then the tally."""
    width = max(len(shown) for shown, _ in _EXERCISES.values())
    print("progress board")
    for label, (shown, _) in _EXERCISES.items():
        state = _STATUS.get(label, "not started")
        print(f"  {_MARKS[state]} {label:<12} {shown:<{width}}  {state}")
    done = sum(_STATUS.get(label) == "passed" for label in _EXERCISES)
    print(f"\n{done} of {len(_EXERCISES)} exercises complete")
    failing = [label for label in _EXERCISES if _STATUS.get(label) == "failed"]
    if failing:
        print("failing right now: " + ", ".join(failing) + ". Each one printed what went "
              "wrong in its own cell above, and every exercise has hints you can open.")
    elif done < len(_EXERCISES):
        print("work top to bottom: every exercise has hints you can open above its code.")


# Your progress board. Every check is re-run here, quietly, against your code as it stands
# now (each one already printed its feedback in its own cell above), so the board is
# current even if you edited an exercise and did not re-run its check.
if __name__ == "__main__":
    with contextlib.redirect_stdout(io.StringIO()):
        for _name, _check, _needs in (("exercise 1", _check_token_bucket, ()),
                                      ("exercise 2", _check_backoff_ms, ()),
                                      ("exercise 3", _check_pacer, ("exercise 1",)),
                                      ("exercise 4", _check_class_capacity, ()),
                                      ("exercise 5", _check_pick_next, ())):
            _try(_name, _check, _needs)
    _progress_board()
    # A stub you have not reached yet is not a failure. A check that ran and came back wrong
    # is: in a script or under CI it ends this run non-zero, rather than letting a green exit
    # code paper over it. Inside a notebook kernel the board above has already said so, in a
    # line rather than a traceback at the foot of the page.
    if _FAILED_CHECKS and "ipykernel" not in sys.modules:
        raise SystemExit("checks failed: " + ", ".join(dict.fromkeys(_FAILED_CHECKS)))

<!-- COMMONS NOTICE v1 · generated by tools/notebooks.py · do not edit by hand -->
---
**Synapsa Commons** · © 2026 RealAI · licensed under [CC BY-NC-SA
4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/)

**You may** use this lesson to learn and to teach, and copy, fork, share and adapt it.

**You must** credit "Synapsa Commons by RealAI" with a link to
https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials,
say what you changed, and share anything you adapt under this same licence.

**You may not** use it, or anything adapted from it, in a way primarily intended for
commercial advantage or payment: for example selling it, charging for a course, bootcamp or
training built on it, or packaging it into a paid product or service. For a commercial
licence, contact [RealAI](https://www.realai.eu/contact).

Third-party material in this lesson keeps its own licence, named in `assets/SOURCE.md` or
`claims.yaml`. The Synapsa name and logo belong to RealAI and are not licensed. This summary
is not the licence: the [legal
code](https://creativecommons.org/licenses/by-nc-sa/4.0/legalcode) governs.